# Step 5 — Final L4 Tracking
## Okutama | 3 Detectors × 2 Trackers | Global-ID Recovery

Final configuration:
- Detector thresholds: YOLO26s=0.20, RT-DETR-R18=0.45, BPD=0.20
- Short-term lost-track retention: 3 seconds
- Long-term Global-ID ReID recovery: enabled
- BoT-SORT internal ReID + sparseOptFlow GMC
- 5 videos per run
- Video selection is randomized reproducibly by `VIDEO_SELECTION_SEED`
- Changing only `VIDEO_SELECTION_SEED` selects a different reproducible 5-video set
- Output folder automatically includes the seed to prevent cache/result mixing


In [25]:
# 1. Mount Google Drive
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if not IN_COLAB:
    raise RuntimeError("Run this controlled notebook in Google Colab.")

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
MOUNTED_DRIVE_ROOT = Path("/content/drive/MyDrive")
if not MOUNTED_DRIVE_ROOT.exists():
    raise FileNotFoundError(MOUNTED_DRIVE_ROOT)
print("Mounted output Drive:", MOUNTED_DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mounted output Drive: /content/drive/MyDrive


## 2. Install controlled dependencies
PyTorch/CUDA from Colab are preserved. The remaining dependencies are installed before
loading tracker/model code.

In [26]:
import subprocess
import sys

PACKAGES = [
    "ultralytics==8.4.116",
    "lap==0.5.13",
    "gdown==6.1.0",
    "onnx>=1.17.0",
    "PyYAML>=6.0",
    "pandas>=2.0",
    "numpy>=1.26",
    "scipy>=1.11",
    "tqdm>=4.66",
    "pillow>=10.0",
    "requests>=2.31",
    "matplotlib>=3.8",
    "onnxruntime-gpu==1.24.4",
    "faster-coco-eval>=1.6.7",
    "nvidia-ml-py>=12.560.30",
    "tabulate>=0.9",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed", *PACKAGES
])


import importlib

try:
    import tensorrt as trt
except Exception:
    import torch as _torch_for_trt_install

    cuda_text = str(_torch_for_trt_install.version.cuda or "")
    package = "tensorrt-cu12" if cuda_text.startswith("12") else "tensorrt"

    print(f"TensorRT Python package not found. Installing {package}...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        package,
    ])

    importlib.invalidate_caches()
    import tensorrt as trt

import cv2
import gdown
import onnx
import numpy as np
import pandas as pd
import requests
import scipy
import torch
import torchvision
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select an NVIDIA GPU runtime in Colab.")

import logging
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*['\"]half['\"] is deprecated.*",
)

try:
    from ultralytics.utils import LOGGER as ULTRALYTICS_LOGGER

    class _SuppressUltralyticsHalfDeprecation(logging.Filter):
        def filter(self, record):
            try:
                message = record.getMessage()
            except Exception:
                return True

            return not (
                "'half' is deprecated" in message
                or '"half" is deprecated' in message
            )

    ULTRALYTICS_LOGGER.addFilter(
        _SuppressUltralyticsHalfDeprecation()
    )

except Exception:
    pass

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA        :", torch.version.cuda)
print("Ultralytics :", ultralytics.__version__)
print("gdown       :", getattr(gdown, "__version__", "unknown"))
print("ONNX        :", onnx.__version__)
print("TensorRT    :", trt.__version__)
print("OpenCV      :", cv2.__version__)
print("GPU         :", torch.cuda.get_device_name(0))


Python      : 3.12.13
PyTorch     : 2.11.0+cu128
Torchvision : 0.26.0+cu128
CUDA        : 12.8
Ultralytics : 8.4.116
gdown       : 6.1.0
ONNX        : 1.22.0
TensorRT    : 11.2.1.2
OpenCV      : 5.0.0
GPU         : NVIDIA L4


## 3. USER CONFIGURATION — edit only this cell
Paste the three final model paths or shared-file URLs below.

Supported source examples:
- `/content/drive/MyDrive/.../best.pt`
- `https://drive.google.com/file/d/.../view?usp=sharing`
- a generic direct `https://.../model.pt` URL

`TEST_4K` is the recommended final Okutama run (official test-video archive, about 4 GB).
`SAMPLE_4K` is a faster one-video smoke/full-pipeline run (about 540 MB).
`CUSTOM` accepts a local folder/ZIP or a downloadable ZIP in OKUTAMA_CUSTOM_SOURCE.

In [27]:
# --------------------------- MODEL SOURCES ---------------------------
YOLO26S_MODEL_SOURCE = "https://drive.google.com/file/d/10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r/view?usp=drive_link"
RTDETR_MODEL_SOURCE = "https://drive.google.com/file/d/10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u/view?usp=drive_link"
BPD_MODEL_SOURCE = "https://drive.google.com/file/d/1cbf-pONCQaaEtzLndOxPxomltRDjmv2a/view?usp=drive_link"

# Model format handling.
# "auto" is strongly recommended for .pt/.pth/.onnx links.
# Use "engine" explicitly only when the source really is a TensorRT engine.
MODEL_FORMAT_HINTS = {
    "YOLO26s": "auto",
    "RT-DETR-R18": "auto",
    "BPD-YOLOn/L-FPN": "auto",
}

# Optional filename substring filters.
# Leave empty for normal direct file links.
# If you paste a Google Drive FOLDER link containing several model files,
# set a unique substring here so the intended file can be selected.
MODEL_FILENAME_CONTAINS = {
    "YOLO26s": "",
    "RT-DETR-R18": "",
    "BPD-YOLOn/L-FPN": "",
}

# Never trust an old cache merely because its file size is non-zero.
# Files are validated before reuse.
VALIDATE_MODEL_BINARY_BEFORE_RUN = True


# --------------------------- VIDEO SELECTION ---------------------------
# Change ONLY this value to select a different reproducible 5-video set.
VIDEO_SELECTION_SEED = 42

# Number of Okutama videos evaluated in each run.
MAX_VIDEOS = 5


# --------------------------- OUTPUT ---------------------------
OUTPUT_RELATIVE_DIR = "aerial_human_detection/step5_tracking_final"

# RUN_TAG automatically changes when VIDEO_SELECTION_SEED changes.
# Example:
#   seed 44  -> okutama_l4_final_s44
#   seed 100 -> okutama_l4_final_s100
RUN_TAG = f"okutama_l4_final_s{VIDEO_SELECTION_SEED}"


# --------------------------- HARDWARE / DETECTOR ---------------------------
REQUIRE_L4 = True
DEVICE_ID = 0

IMAGE_SIZE = 1280
MAX_DETECTIONS = 3000

DETECTOR_CONF_THRESHOLDS = {
    "YOLO26s": 0.20,
    "RT-DETR-R18": 0.45,
    "BPD-YOLOn/L-FPN": 0.20,
}

DETECTOR_NMS_IOU = 0.70
DETECTION_EVAL_IOU = 0.50

USE_FP16_CHECKPOINT_INFERENCE = False

# The deprecated Ultralytics half predict option is intentionally not passed.
# TensorRT engines keep their exported precision automatically.
WARMUP_RUNS = 3


# --------------------------- TRACKERS ---------------------------
#
# ID-STABLE PROFILE
#
# TRACK_LOST_GRACE_SECONDS does NOT forcibly lock an identity.
# It keeps a lost track alive long enough to be re-associated before it
# is removed.
#
# Ultralytics 8.4.116 uses track_buffer directly as a frame count, so the
# effective buffer is calculated dynamically as:
#
#       round(video_fps * TRACK_LOST_GRACE_SECONDS)
#
# Example:
#       30 FPS × 3 seconds = 90 frames
#

TRACKERS_TO_RUN = [
    "ByteTrack",
    "BoT-SORT",
]

TRACK_LOST_GRACE_SECONDS = 3.0


# Shared association policy.
TRACKER_COMMON_SETTINGS = {
    "match_thresh": 0.85,
    "fuse_score": True,
}


# Model-aware tracker confidence gates.
#
# These are tracker association thresholds, NOT detector thresholds.
TRACKER_MODEL_THRESHOLDS = {
    "YOLO26s": {
        "track_high_thresh": 0.28,
        "track_low_thresh": 0.18,
        "new_track_thresh": 0.32,
    },

    "RT-DETR-R18": {
        "track_high_thresh": 0.50,
        "track_low_thresh": 0.40,
        "new_track_thresh": 0.52,
    },

    "BPD-YOLOn/L-FPN": {
        "track_high_thresh": 0.28,
        "track_low_thresh": 0.18,
        "new_track_thresh": 0.32,
    },
}


# ByteTrack keeps motion/IoU association only.
BYTETRACK_SETTINGS = {
    **TRACKER_COMMON_SETTINGS,
}


# BoT-SORT:
# - GMC for moving aerial cameras.
# - ReID for stronger identity consistency.
BOTSORT_SETTINGS = {
    **TRACKER_COMMON_SETTINGS,

    "gmc_method": "sparseOptFlow",

    "proximity_thresh": 0.35,
    "appearance_thresh": 0.85,

    "with_reid": True,

    "model": "yolo26n-reid.onnx",

    "device": f"cuda:{DEVICE_ID}",
}


# Tracking quality is the Step-5 objective.
# Runtime remains internal only; Step 4 is the deployment benchmark.
REPORT_RUNTIME_METRICS_IN_STEP5 = False


# --------------------------- LONG-TERM GLOBAL ID RECOVERY ---------------------------
#
# Short gaps:
#       handled by TRACK_LOST_GRACE_SECONDS
#
# Complete FOV exits / long gaps:
#       handled by Long-Term Global-ID appearance memory
#

ENABLE_GLOBAL_ID_RECOVERY = True


# ReID encoder.
GLOBAL_REID_MODEL = "yolo26n-reid.onnx"


# Keep inactive identities in memory long enough for camera
# pan-away / pan-back events.
GLOBAL_ID_MEMORY_SECONDS = 60.0


# Conservative recovery gates.
# A false recovery is considered worse than creating a new Global ID.
GLOBAL_REID_MIN_COSINE = 0.88
GLOBAL_REID_MIN_COMBINED_SCORE = 0.83
GLOBAL_REID_MIN_MARGIN = 0.06


# Appearance prototype update.
GLOBAL_REID_EMA_ALPHA = 0.90

# ReID embedding does not need to be recomputed on every frame.
GLOBAL_REID_FEATURE_INTERVAL_FRAMES = 5


# Extremely small crops do not provide reliable appearance information.
GLOBAL_REID_MIN_BOX_HEIGHT_PX = 12
GLOBAL_REID_MIN_BOX_WIDTH_PX = 5


# Combined Global-ID matching weights.
GLOBAL_REID_WEIGHT_APPEARANCE = 0.80
GLOBAL_REID_WEIGHT_COLOR = 0.08
GLOBAL_REID_WEIGHT_SIZE = 0.07
GLOBAL_REID_WEIGHT_TIME = 0.05


# Small optional border cue.
# It is NOT a hard restriction because aerial camera motion can cause
# a returning subject to reappear anywhere in the image.
GLOBAL_REID_BORDER_BONUS = 0.02
GLOBAL_REID_BORDER_MARGIN_RATIO = 0.08


# If True, the experiment stops if the Global ReID encoder cannot load.
# This prevents silently evaluating a different pipeline.
GLOBAL_REID_REQUIRED = True


# --------------------------- OKUTAMA ---------------------------
DATASET_PROFILE = "TEST_4K"       # Final controlled run
# DATASET_PROFILE = "SAMPLE_4K"   # Quick run
# DATASET_PROFILE = "CUSTOM"

OKUTAMA_CUSTOM_SOURCE = ""

PREFER_DISTINCT_SCENARIOS = True

SEGMENTS_PER_VIDEO = 1
SEGMENT_LENGTH_FRAMES = 300
SEGMENT_STRIDE_FRAMES = 300

MIN_MEAN_PERSONS_PER_FRAME = 1.0


# --------------------------- EXPORT / RESUME ---------------------------
SAVE_ANNOTATED_VIDEO = True

SHOW_GROUND_TRUTH_ON_VIDEO = True
SHOW_TRACK_TRAILS = True
TRACK_TRAIL_LENGTH = 30

SAVE_DETECTION_CACHE_JSONL = True
SAVE_FRAME_CSV = True
SAVE_TRACK_JSONL = True

RESUME_COMPLETED_DETECTION_CACHE = True
RESUME_COMPLETED_TRACKER_RUNS = True

CONTINUE_ON_SYSTEM_ERROR = False
FAIL_IF_FINAL_MATRIX_INCOMPLETE = True


# --------------------------- PROPOSAL REFERENCE CHECKS ---------------------------
# These are reporting gates, not tuning targets on Okutama.

TARGET_MOTA = 0.50
TARGET_IDF1 = 0.60
TARGET_IDSW_PER_100_FRAMES = 5.0


# --------------------------- PINNED / PRIMARY SOURCES ---------------------------
RTDETR_REPO_URL = "https://github.com/lyuwenyu/RT-DETR.git"

RTDETR_REPO_COMMIT = (
    "199fc382f53abbfb5c1804c97b0e8b204e3cb8d0"
)

TRACKEVAL_REPO_URL = (
    "https://github.com/JonathonLuiten/TrackEval.git"
)


OKUTAMA_OFFICIAL_URLS = {
    "SAMPLE_4K": (
        "https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/"
        "AKx_1WK7YqOIf4I0PDX5vRk/Sample.zip"
        "?dl=1&e=1&rlkey=7u7131amaul29amyr4jbnnu03"
    ),

    "TEST_4K": (
        "https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/"
        "AKym5JciNZCmrCHfQkfoBtU/TestSetVideos.zip"
        "?dl=1&e=1&rlkey=7u7131amaul29amyr4jbnnu03"
    ),
}


# --------------------------- CONFIGURATION SUMMARY ---------------------------
print("=" * 96)
print("STEP-5 CONFIGURATION")
print("=" * 96)

print("Dataset profile     :", DATASET_PROFILE)

print("Video selection seed:", VIDEO_SELECTION_SEED)
print("Maximum videos      :", MAX_VIDEOS)

print("Segments per video  :", SEGMENTS_PER_VIDEO)
print("Frames per segment  :", SEGMENT_LENGTH_FRAMES)

print("Image size          :", IMAGE_SIZE)

print("Detector confidence thresholds:")
for _model_name, _threshold in DETECTOR_CONF_THRESHOLDS.items():
    print(
        f"  {_model_name:20s}: "
        f"{_threshold:.2f}"
    )

print("Trackers            :", TRACKERS_TO_RUN)

print(
    "Lost-track grace    :",
    f"{TRACK_LOST_GRACE_SECONDS:.1f} seconds",
)

print(
    "BoT-SORT ReID       :",
    BOTSORT_SETTINGS["with_reid"],
)

print(
    "BoT-SORT ReID model :",
    BOTSORT_SETTINGS["model"],
)

print(
    "Global-ID recovery  :",
    ENABLE_GLOBAL_ID_RECOVERY,
)

print(
    "Global ReID model   :",
    GLOBAL_REID_MODEL,
)

print(
    "Global memory       :",
    f"{GLOBAL_ID_MEMORY_SECONDS:.0f} seconds",
)

print(
    "Global cosine gate  :",
    GLOBAL_REID_MIN_COSINE,
)

print("Run tag             :", RUN_TAG)
print("Require NVIDIA L4   :", REQUIRE_L4)

print("=" * 96)

STEP-5 CONFIGURATION
Dataset profile     : TEST_4K
Video selection seed: 42
Maximum videos      : 5
Segments per video  : 1
Frames per segment  : 300
Image size          : 1280
Detector confidence thresholds:
  YOLO26s             : 0.20
  RT-DETR-R18         : 0.45
  BPD-YOLOn/L-FPN     : 0.20
Trackers            : ['ByteTrack', 'BoT-SORT']
Lost-track grace    : 3.0 seconds
BoT-SORT ReID       : True
BoT-SORT ReID model : yolo26n-reid.onnx
Global-ID recovery  : True
Global ReID model   : yolo26n-reid.onnx
Global memory       : 60 seconds
Global cosine gate  : 0.88
Run tag             : okutama_l4_final_s42
Require NVIDIA L4   : True


## How to change the 5 test videos

From this version onward, **only change this one value** in the configuration cell:

```python
VIDEO_SELECTION_SEED = 42
```

Examples:

```python
VIDEO_SELECTION_SEED = 100
VIDEO_SELECTION_SEED = 2026
VIDEO_SELECTION_SEED = 777
```

Each seed produces a different reproducible 5-video set. The output folder also changes automatically:

```text
okutama_l4_final_s42
okutama_l4_final_s100
okutama_l4_final_s2026
```

The exact selected videos are saved to:

```text
selected_segments_seeded.csv
video_selection_config.json
```


## Final identity-stability policy

This final version uses two complementary mechanisms:

- **Short-term continuity:** `track_buffer = round(video_fps × 3 seconds)`.
- **Long-term continuity:** an appearance gallery stores Global Person IDs for up to 60 seconds
  and can reconnect a new Local Track ID after the person has completely left the field of view.

For BoT-SORT, internal ReID and sparse-optical-flow camera-motion compensation remain enabled.
For ByteTrack, the external Global-ID layer supplies the long-term appearance memory that standard
ByteTrack does not provide internally.

The Global-ID matcher is deliberately conservative. A previous identity is restored only when the
appearance similarity, combined evidence, and best-vs-second-best margin all pass fixed gates.
Otherwise a new Global ID is created.

**Evaluation integrity:** these settings were selected after an earlier Okutama pilot. For an
unbiased final claim, freeze this configuration and evaluate it on fresh clips/videos that were not
used to choose these settings.


## Final operating thresholds and warning policy

The final L4 run uses fixed detector confidence thresholds:

- **YOLO26s:** `0.20`
- **RT-DETR-R18:** `0.45`
- **BPD-YOLOn/L-FPN:** `0.20`

These thresholds are applied before tracker association and are saved in the run audit.

The deprecated Ultralytics half predict option is not passed anywhere in this version.
TensorRT engines retain the precision with which they were exported. A narrow log filter also
suppresses only that specific legacy deprecation message if an internal Ultralytics layer emits it;
other warnings remain visible.

Because the confidence thresholds changed from the earlier `0.08` run, this version uses a new
`RUN_TAG`, so old detection caches cannot be reused accidentally.


## 4. Runtime audit, persistent output tree, and shared helpers

In [28]:

import contextlib
import gc
import hashlib
import importlib
import json
import math
import os
import platform
import re
import shlex
import shutil
import time
import traceback
import urllib.parse
import zipfile
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from IPython.display import display

OUTPUT_ROOT = MOUNTED_DRIVE_ROOT / OUTPUT_RELATIVE_DIR / RUN_TAG
AUDIT_DIR = OUTPUT_ROOT / "00_audit"
MODEL_META_DIR = OUTPUT_ROOT / "01_models"
DATA_META_DIR = OUTPUT_ROOT / "02_dataset"
RUNS_DIR = OUTPUT_ROOT / "03_runs"
TRACKEVAL_DIR = OUTPUT_ROOT / "04_trackeval"
METRICS_DIR = OUTPUT_ROOT / "05_metrics"
PLOTS_DIR = OUTPUT_ROOT / "06_plots"
REPORT_ASSETS_DIR = OUTPUT_ROOT / "07_report_assets"

for p in [OUTPUT_ROOT, AUDIT_DIR, MODEL_META_DIR, DATA_META_DIR, RUNS_DIR,
          TRACKEVAL_DIR, METRICS_DIR, PLOTS_DIR, REPORT_ASSETS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_CACHE = Path("/content/step5_model_cache_v5")
DATA_CACHE = Path("/content/step5_okutama_cache")
TOOL_CACHE = Path("/content/step5_tools")
for p in [MODEL_CACHE, DATA_CACHE, TOOL_CACHE]:
    p.mkdir(parents=True, exist_ok=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def safe_slug(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))
    value = re.sub(r"_+", "_", value).strip("_.")
    return value or "item"


def percentile(values: Sequence[float], q: float) -> float:
    arr = np.asarray(list(values), dtype=np.float64)
    return float(np.percentile(arr, q)) if arr.size else float("nan")


def nanmean(values: Sequence[float]) -> float:
    arr = np.asarray(list(values), dtype=np.float64)
    return float(np.nanmean(arr)) if arr.size else float("nan")


def is_http_url(value: str) -> bool:
    return isinstance(value, str) and value.lower().startswith(("http://", "https://"))


def is_google_drive_url(value: str) -> bool:
    return is_http_url(value) and ("drive.google.com" in value.lower() or "docs.google.com" in value.lower())


def is_placeholder(value: str) -> bool:
    text = str(value).strip().upper()
    return (not text) or text.startswith("PASTE_") or "PUT_YOUR" in text


def detector_conf_threshold(model_name: str) -> float:
    """Return the fixed Step-5 confidence threshold for a detector."""
    if model_name not in DETECTOR_CONF_THRESHOLDS:
        raise KeyError(
            f"No detector confidence threshold configured for: {model_name}"
        )

    return float(DETECTOR_CONF_THRESHOLDS[model_name])


def cuda_sync() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize(DEVICE_ID)


def reset_cuda_peak() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE_ID)


def peak_cuda_mib() -> float:
    return float(torch.cuda.max_memory_allocated(DEVICE_ID) / (1024 ** 2)) if torch.cuda.is_available() else 0.0



def normalize_download_url(url: str) -> str:
    """
    Normalize cloud URLs for direct binary download.
    Dropbox links are forced to dl=1.
    """
    url = str(url).strip()

    if not is_http_url(url):
        return url

    parsed = urllib.parse.urlsplit(url)

    if "dropbox.com" in parsed.netloc.lower():
        query = urllib.parse.parse_qs(
            parsed.query,
            keep_blank_values=True,
        )
        query["dl"] = ["1"]

        flat_query = {
            key: values[-1]
            for key, values in query.items()
        }

        url = urllib.parse.urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                parsed.path,
                urllib.parse.urlencode(flat_query),
                parsed.fragment,
            )
        )

    return url


def http_download_robust(
    url: str,
    destination: Path,
    minimum_bytes: int = 1_000_000,
) -> Path:
    """
    Robust large-file downloader for Colab.

    - follows redirects
    - retries transient failures
    - resumes from a .part file when supported
    - retries from scratch if resume is rejected
    - atomically finalizes the download
    - rejects tiny HTML/error responses
    """
    destination = Path(destination)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    url = normalize_download_url(url)

    part_path = destination.with_name(
        destination.name + ".part"
    )

    print("Resolved download URL:", url)
    print("Destination          :", destination)

    resume_cmd = [
        "curl",
        "-L",
        "--fail",
        "--show-error",
        "--retry",
        "5",
        "--retry-delay",
        "3",
        "--retry-all-errors",
        "--connect-timeout",
        "30",
        "-C",
        "-",
        "-o",
        str(part_path),
        url,
    ]

    result = subprocess.run(
        resume_cmd,
        check=False,
    )

    if result.returncode != 0:
        print(
            "Resume mode failed or is unsupported. "
            "Retrying the download from scratch..."
        )

        if part_path.exists():
            part_path.unlink()

        fresh_cmd = [
            "curl",
            "-L",
            "--fail",
            "--show-error",
            "--retry",
            "5",
            "--retry-delay",
            "3",
            "--retry-all-errors",
            "--connect-timeout",
            "30",
            "-o",
            str(part_path),
            url,
        ]

        subprocess.check_call(
            fresh_cmd
        )

    if not part_path.exists():
        raise RuntimeError(
            f"Download finished without creating: {part_path}"
        )

    size_bytes = part_path.stat().st_size

    print(
        "Downloaded size      : "
        f"{size_bytes:,} bytes "
        f"({size_bytes / (1024 ** 2):.2f} MiB)"
    )

    if size_bytes < int(minimum_bytes):
        with part_path.open("rb") as f:
            preview_bytes = f.read(2048)

        preview = preview_bytes.decode(
            "utf-8",
            errors="replace",
        )[:500]

        raise RuntimeError(
            "Downloaded response is unexpectedly small and "
            "is probably an HTML/error response rather than "
            "the requested binary file. "
            f"size_bytes={size_bytes:,}; preview={preview!r}"
        )

    if destination.exists():
        destination.unlink()

    part_path.replace(
        destination
    )

    return destination


GPU_NAME = torch.cuda.get_device_name(DEVICE_ID)
if REQUIRE_L4 and "L4" not in GPU_NAME.upper():
    raise RuntimeError(f"Controlled Step-5 requires NVIDIA L4; active GPU: {GPU_NAME}")

environment = {
    "timestamp_utc": utc_now_iso(),
    "platform": platform.platform(),
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_total_memory_gib": torch.cuda.get_device_properties(DEVICE_ID).total_memory / (1024 ** 3),
    "ultralytics": ultralytics.__version__,
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "rtdetr_repo": RTDETR_REPO_URL,
    "rtdetr_commit": RTDETR_REPO_COMMIT,
    "trackeval_repo": TRACKEVAL_REPO_URL,
    "model_source_hotfix": "v5_tensorrt_plus_okutama_download_fix",
}
write_json(AUDIT_DIR / "environment.json", environment)

with (AUDIT_DIR / "pip_freeze.txt").open("w", encoding="utf-8") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, stderr=subprocess.STDOUT, text=True, check=False)
with (AUDIT_DIR / "nvidia_smi.txt").open("w", encoding="utf-8") as f:
    subprocess.run(["nvidia-smi"], stdout=f, stderr=subprocess.STDOUT, text=True, check=False)

execution_config = {
    "output_root": str(OUTPUT_ROOT),
    "run_tag": RUN_TAG,
    "models": ["YOLO26s", "RT-DETR-R18", "BPD-YOLOn/L-FPN"],
    "trackers": TRACKERS_TO_RUN,
    "image_size": IMAGE_SIZE,
    "detector_conf_thresholds": DETECTOR_CONF_THRESHOLDS,
    "detector_nms_iou": DETECTOR_NMS_IOU,
    "max_detections": MAX_DETECTIONS,
    "detection_eval_iou": DETECTION_EVAL_IOU,
    "dataset_profile": DATASET_PROFILE,
    "max_videos": MAX_VIDEOS,
    "segments_per_video": SEGMENTS_PER_VIDEO,
    "segment_length_frames": SEGMENT_LENGTH_FRAMES,
    "segment_stride_frames": SEGMENT_STRIDE_FRAMES,
    "bytetrack_base_settings": BYTETRACK_SETTINGS,
    "botsort_base_settings": BOTSORT_SETTINGS,
    "tracker_model_thresholds": TRACKER_MODEL_THRESHOLDS,
    "track_lost_grace_seconds": TRACK_LOST_GRACE_SECONDS,
    "botsort_reid_enabled": bool(BOTSORT_SETTINGS.get("with_reid", False)),
    "botsort_reid_model": BOTSORT_SETTINGS.get("model"),
    "tracker_settings_selected_after_okutama_pilot": True,
    "global_id_recovery_enabled": ENABLE_GLOBAL_ID_RECOVERY,
    "global_reid_model": GLOBAL_REID_MODEL,
    "global_id_memory_seconds": GLOBAL_ID_MEMORY_SECONDS,
    "global_reid_min_cosine": GLOBAL_REID_MIN_COSINE,
    "global_reid_min_combined_score": GLOBAL_REID_MIN_COMBINED_SCORE,
    "global_reid_min_margin": GLOBAL_REID_MIN_MARGIN,
    "global_reid_feature_interval_frames": GLOBAL_REID_FEATURE_INTERVAL_FRAMES,
    "global_reid_required": GLOBAL_REID_REQUIRED,
    "private_final_test_used": False,
}
write_json(AUDIT_DIR / "execution_config.json", execution_config)
print("GPU        :", GPU_NAME)
print("Output root:", OUTPUT_ROOT)


GPU        : NVIDIA L4
Output root: /content/drive/MyDrive/aerial_human_detection/step5_tracking_final/okutama_l4_final_s42


## 5. Resolve and verify the three model artifacts — v4 TensorRT support

The supplied Google Drive links are the TensorRT artifacts produced in Step 4.

v4 supports:
- PyTorch checkpoint (`.pt` / `.pth`)
- ONNX (`.onnx`)
- TensorRT (`.engine`)

It recognizes both TensorRT containers used by this project:
- Ultralytics metadata-wrapped engine for YOLO26s and BPD-YOLOn/L-FPN.
- Raw TensorRT `ftrt` engine for RT-DETR-R18.

For detected TensorRT engines, SHA-256 is compared with the hashes recorded in the Step-4 report.


In [29]:
MODEL_SOURCES = {
    "YOLO26s": YOLO26S_MODEL_SOURCE,
    "RT-DETR-R18": RTDETR_MODEL_SOURCE,
    "BPD-YOLOn/L-FPN": BPD_MODEL_SOURCE,
}

MODEL_DEFAULT_BASENAMES = {
    "YOLO26s": "yolo26s_step5_model",
    "RT-DETR-R18": "rtdetr_r18_step5_model",
    "BPD-YOLOn/L-FPN": "bpd_yolon_lfpn_step5_model",
}

ALLOWED_MODEL_FORMATS = {
    "YOLO26s": {"pt", "onnx", "engine"},
    "RT-DETR-R18": {"pth", "onnx", "engine"},
    "BPD-YOLOn/L-FPN": {"pt", "onnx", "engine"},
}

KNOWN_STEP4_ENGINE_SHA256 = {
    "YOLO26s": "fc46b73eca3520a3dd29b9d187b8fadcdae23ab215af6347602bb382a63b5dfa",
    "RT-DETR-R18": "9d9a2ad34cad42e3b8c80299887552d29729448dda2b278345c3cc3b92257b51",
    "BPD-YOLOn/L-FPN": "518d8aafccc491a17e6c219fa12f112dbedfd60b3b0e343004e5e9b3e2e06112",
}



def _source_hash(source: str) -> str:
    return hashlib.sha256(str(source).strip().encode("utf-8")).hexdigest()[:16]


def _first_bytes(path: Path, n: int = 64) -> bytes:
    with path.open("rb") as f:
        return f.read(n)


def _is_probably_html_or_text_error(path: Path) -> Tuple[bool, str]:
    head = _first_bytes(path, 1024)
    low = head.lower()

    signatures = [
        b"<!doctype html",
        b"<html",
        b"<head",
        b"<body",
        b"access denied",
        b"permission denied",
        b"google drive",
        b"quota exceeded",
        b"virus scan warning",
    ]

    if any(sig in low for sig in signatures):
        preview = head.decode("utf-8", errors="replace")[:300]
        return True, preview

    return False, ""


def _torch_archive_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    """
    Structural Torch checkpoint probe without unpickling custom project classes.
    Modern torch.save files are ZIP archives containing pickle metadata.
    Legacy pickle-based torch files commonly start with pickle protocol 0x80.
    """
    info = {
        "zipfile": False,
        "zip_integrity": None,
        "pickle_metadata": False,
        "legacy_pickle_header": False,
    }

    head = _first_bytes(path, 16)

    # Legacy pickle-based serialization.
    if len(head) >= 1 and head[0] == 0x80:
        info["legacy_pickle_header"] = True
        return True, info

    if not zipfile.is_zipfile(path):
        return False, info

    info["zipfile"] = True

    try:
        with zipfile.ZipFile(path, "r") as zf:
            bad_member = zf.testzip()
            info["zip_integrity"] = (bad_member is None)
            names = zf.namelist()
            info["pickle_metadata"] = any(
                name.endswith(".pkl")
                or name.endswith("data.pkl")
                or "/data.pkl" in name
                for name in names
            )
            info["zip_member_sample"] = names[:10]

            if bad_member is not None:
                info["bad_zip_member"] = bad_member
                return False, info

            if info["pickle_metadata"]:
                return True, info

    except Exception as exc:
        info["zip_error"] = repr(exc)
        return False, info

    return False, info


def _onnx_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    info = {}

    try:
        model = onnx.load(str(path), load_external_data=False)
        info["ir_version"] = int(model.ir_version)
        info["graph_name"] = str(model.graph.name)
        info["num_nodes"] = int(len(model.graph.node))
        info["num_inputs"] = int(len(model.graph.input))
        info["num_outputs"] = int(len(model.graph.output))

        # check_model may raise on malformed/truncated protobuf.
        onnx.checker.check_model(model)
        info["checker"] = "PASSED"

        del model
        return True, info

    except Exception as exc:
        info["error"] = f"{type(exc).__name__}: {exc}"
        return False, info


def _ultralytics_engine_metadata_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    info = {}

    try:
        file_size = path.stat().st_size

        if file_size < 16:
            return False, {"error": "file too small"}

        with path.open("rb") as f:
            length_bytes = f.read(4)

            if len(length_bytes) != 4:
                return False, {"error": "missing metadata length"}

            meta_len = int.from_bytes(
                length_bytes,
                byteorder="little",
                signed=True,
            )
            info["metadata_length"] = int(meta_len)

            if not (1 <= meta_len <= min(16 * 1024 * 1024, file_size - 8)):
                return False, info

            meta_bytes = f.read(meta_len)

            try:
                metadata = json.loads(meta_bytes.decode("utf-8"))
            except Exception as exc:
                info["metadata_json_error"] = f"{type(exc).__name__}: {exc}"
                return False, info

            if not isinstance(metadata, dict):
                return False, info

            info["description"] = str(metadata.get("description", ""))
            info["task"] = metadata.get("task")
            info["imgsz"] = metadata.get("imgsz")
            info["metadata"] = metadata
            info["engine_payload_offset"] = 4 + meta_len
            info["engine_payload_first_16_hex"] = f.read(16).hex()

            looks_ultralytics = (
                "ultralytics" in info["description"].lower()
                or "task" in metadata
                or "imgsz" in metadata
                or "names" in metadata
            )

            return bool(looks_ultralytics), info

    except Exception as exc:
        info["error"] = f"{type(exc).__name__}: {exc}"
        return False, info


def _raw_tensorrt_magic_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    head = _first_bytes(path, 32)
    info = {
        "first_32_bytes_hex": head.hex(),
        "first_4_ascii": head[:4].decode("latin1", errors="replace"),
    }

    if head[:4] == b"ftrt":
        info["magic"] = "ftrt"
        return True, info

    return False, info


def _tensorrt_engine_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    wrapped_ok, wrapped_info = _ultralytics_engine_metadata_probe(path)

    if wrapped_ok:
        return True, {
            "engine_container": "ultralytics_metadata_wrapped",
            "ultralytics_metadata": wrapped_info,
        }

    raw_ok, raw_info = _raw_tensorrt_magic_probe(path)

    if raw_ok:
        return True, {
            "engine_container": "raw_tensorrt",
            "raw_engine": raw_info,
        }

    return False, {
        "ultralytics_metadata_probe": wrapped_info,
        "raw_engine_probe": raw_info,
    }

def detect_model_format(path: Path, requested_hint: str = "auto") -> Tuple[str, Dict[str, Any]]:
    """
    Returns one of: torch, onnx, engine, unknown.
    requested_hint may be auto/pt/pth/onnx/engine.
    """
    requested_hint = str(requested_hint).lower().strip().lstrip(".")
    diagnostics = {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "first_32_bytes_hex": _first_bytes(path, 32).hex(),
        "requested_hint": requested_hint,
    }

    is_text_error, preview = _is_probably_html_or_text_error(path)

    if is_text_error:
        diagnostics["text_error_preview"] = preview
        return "unknown", diagnostics

    if requested_hint in {"pt", "pth", "auto"}:
        torch_ok, torch_info = _torch_archive_probe(path)
        diagnostics["torch_probe"] = torch_info

        if torch_ok:
            return "torch", diagnostics

    if requested_hint in {"onnx", "auto"}:
        onnx_ok, onnx_info = _onnx_probe(path)
        diagnostics["onnx_probe"] = onnx_info

        if onnx_ok:
            return "onnx", diagnostics

    if requested_hint in {"engine", "auto"}:
        engine_ok, engine_info = _tensorrt_engine_probe(path)
        diagnostics["engine_probe"] = engine_info

        if engine_ok:
            return "engine", diagnostics

    return "unknown", diagnostics


def canonical_extension(model_name: str, detected_format: str) -> str:
    if detected_format == "onnx":
        return "onnx"

    if detected_format == "engine":
        return "engine"

    if detected_format == "torch":
        return "pth" if model_name == "RT-DETR-R18" else "pt"

    raise ValueError(detected_format)


def _candidate_files(folder: Path) -> List[Path]:
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file()
        and p.stat().st_size >= 100_000
        and not p.name.endswith(".part")
    )


def extract_google_drive_file_id(url: str) -> Optional[str]:
    """
    Extract a Google Drive FILE id from common share-link forms.

    Supported examples:
      https://drive.google.com/file/d/FILE_ID/view?usp=drive_link
      https://drive.google.com/open?id=FILE_ID
      https://drive.google.com/uc?id=FILE_ID
    """
    text = str(url).strip()

    patterns = [
        r"/file/d/([A-Za-z0-9_-]+)",
        r"[?&]id=([A-Za-z0-9_-]+)",
        r"/d/([A-Za-z0-9_-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    return None


def extract_google_drive_folder_id(url: str) -> Optional[str]:
    match = re.search(
        r"/folders/([A-Za-z0-9_-]+)",
        str(url).strip(),
    )
    return match.group(1) if match else None


def _download_google_drive(source: str, download_dir: Path) -> List[Path]:
    """
    v3 hotfix:
    Never download the /view page itself.

    For a Drive FILE URL:
      1. extract FILE_ID ourselves;
      2. call gdown.download(id=FILE_ID, output=explicit_binary_path);
      3. reject tiny HTML/permission pages;
      4. let binary preflight determine pt/pth/onnx.

    For a Drive FOLDER URL:
      use gdown.download_folder(id=FOLDER_ID, ...).
    """
    download_dir.mkdir(parents=True, exist_ok=True)

    folder_id = extract_google_drive_folder_id(source)

    if folder_id:
        print("Google Drive folder ID:", folder_id)

        try:
            result = gdown.download_folder(
                id=folder_id,
                output=str(download_dir),
                quiet=False,
            )
        except Exception as exc:
            raise RuntimeError(
                "Google Drive folder download failed. "
                "Make sure the folder is shared with this Colab session. "
                f"Original error: {type(exc).__name__}: {exc}"
            ) from exc

        files = _candidate_files(download_dir)

        if not files:
            raise RuntimeError(
                "Google Drive folder was resolved, but no model-sized file "
                "was downloaded. Check folder permissions and contents."
            )

        return files

    file_id = extract_google_drive_file_id(source)

    if not file_id:
        raise RuntimeError(
            "Could not extract a Google Drive FILE ID from this URL: "
            f"{source}"
        )

    print("Google Drive file ID:", file_id)

    # Explicit neutral extension. We detect the real binary format afterwards.
    target = download_dir / "google_drive_model_download.bin"

    # Remove any stale/interrupted artifact before a fresh download.
    for stale in [
        target,
        target.with_suffix(target.suffix + ".part"),
    ]:
        if stale.exists():
            stale.unlink()

    try:
        result = gdown.download(
            id=file_id,
            output=str(target),
            quiet=False,
        )
    except Exception as exc:
        raise RuntimeError(
            "Google Drive file download failed. "
            "The file must be accessible to this Colab session; for a public "
            "share link, set sharing to 'Anyone with the link'. "
            f"File ID: {file_id}. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc

    if result is None:
        raise RuntimeError(
            "gdown returned no downloaded file. "
            f"Google Drive file ID: {file_id}"
        )

    if not target.exists():
        result_path = Path(str(result))
        if result_path.exists():
            target = result_path

    if not target.exists():
        raise RuntimeError(
            "gdown reported success but the expected downloaded file "
            f"does not exist. File ID: {file_id}"
        )

    size_bytes = target.stat().st_size
    print(f"Downloaded bytes: {size_bytes:,}")

    # These project checkpoints are many MiB. A tiny response is almost
    # certainly an HTML/access/error response rather than the model.
    if size_bytes < 100_000:
        head = _first_bytes(target, 2048)
        preview = head.decode("utf-8", errors="replace")[:500]

        raise RuntimeError(
            "Google Drive returned a tiny file instead of the model "
            f"({size_bytes:,} bytes). "
            "This is usually a permission/login/share-page response. "
            "Set the file sharing permission to 'Anyone with the link' "
            "or use an existing path in the currently mounted Drive. "
            f"File ID: {file_id}. "
            f"Response preview: {preview!r}"
        )

    return [target]

def _download_generic_url(source: str, download_dir: Path) -> List[Path]:
    download_dir.mkdir(parents=True, exist_ok=True)

    # gdown also supports ordinary HTTP/HTTPS URLs and handles redirects.
    old_cwd = Path.cwd()

    try:
        os.chdir(download_dir)
        result = gdown.download(
            url=source,
            output=None,
            quiet=False,
        )
    finally:
        os.chdir(old_cwd)

    if result is not None:
        result_path = Path(result)

        if not result_path.is_absolute():
            result_path = download_dir / result_path.name

        if result_path.exists():
            return [result_path]

    # Fallback to the project's robust HTTP downloader.
    parsed_name = Path(urllib.parse.urlsplit(source).path).name
    fallback_name = parsed_name if parsed_name else "downloaded_model.bin"
    fallback = download_dir / fallback_name

    http_download_robust(source, fallback)
    return [fallback]


def _select_and_validate_candidate(
    model_name: str,
    source: str,
    candidates: List[Path],
) -> Tuple[Path, str, Dict[str, Any]]:
    hint = str(MODEL_FORMAT_HINTS[model_name]).lower().strip().lstrip(".")
    filename_filter = str(MODEL_FILENAME_CONTAINS[model_name]).strip().lower()

    if hint not in {"auto", "pt", "pth", "onnx", "engine"}:
        raise ValueError(
            f"{model_name}: MODEL_FORMAT_HINTS must be auto/pt/pth/onnx/engine; got {hint}"
        )

    candidates = [
        p for p in candidates
        if p.exists() and p.is_file() and p.stat().st_size >= 100_000
    ]

    if filename_filter:
        candidates = [
            p for p in candidates
            if filename_filter in p.name.lower()
        ]

    if not candidates:
        raise RuntimeError(
            f"{model_name}: no model-sized file was found after resolving source: {source}"
        )

    valid = []
    invalid = []

    for candidate in candidates:
        detected, diag = detect_model_format(candidate, hint)

        if detected == "torch":
            allowed = (
                "pth" in ALLOWED_MODEL_FORMATS[model_name]
                or "pt" in ALLOWED_MODEL_FORMATS[model_name]
            )
        else:
            allowed = detected in ALLOWED_MODEL_FORMATS[model_name]

        if detected != "unknown" and allowed:
            valid.append((candidate, detected, diag))
        else:
            invalid.append({
                "candidate": str(candidate),
                "detected_format": detected,
                "diagnostics": diag,
            })

    if len(valid) == 0:
        diagnostic_path = MODEL_META_DIR / f"{safe_slug(model_name)}_invalid_source_diagnostics.json"
        write_json(
            diagnostic_path,
            {
                "model": model_name,
                "source": source,
                "candidates": invalid,
            },
        )

        first = invalid[0] if invalid else {}
        raise RuntimeError(
            f"{model_name}: downloaded/resolved bytes are not a valid supported model file. "
            f"Diagnostics were saved to {diagnostic_path}. "
            f"First candidate: {first.get('candidate')}; "
            f"first bytes: {first.get('diagnostics', {}).get('first_32_bytes_hex')}. "
            "Typical causes: wrong Google Drive link, private/unshared file, a folder link "
            "containing multiple files, or a model format that does not match the configured hint."
        )

    if len(valid) > 1:
        names = [str(item[0]) for item in valid]
        raise RuntimeError(
            f"{model_name}: more than one valid model candidate was found: {names}. "
            "Set MODEL_FILENAME_CONTAINS for this model to a unique part of the intended filename."
        )

    return valid[0]


def resolve_model_source(model_name: str, source: str) -> Tuple[Path, Dict[str, Any]]:
    source = str(source).strip()

    if is_placeholder(source):
        raise ValueError(
            f"{model_name}: source is still a placeholder. "
            "Edit USER CONFIGURATION and paste the final model path or URL."
        )

    source_cache = MODEL_CACHE / safe_slug(model_name) / _source_hash(source)
    source_cache.mkdir(parents=True, exist_ok=True)

    local_candidate = Path(os.path.expanduser(source))

    if local_candidate.exists():
        if local_candidate.is_dir():
            candidates = _candidate_files(local_candidate)
        else:
            candidates = [local_candidate]

        source_mode = "local_path"

    elif is_google_drive_url(source):
        # Reuse v2 cache only after binary validation.
        cached = _candidate_files(source_cache)

        if cached:
            try:
                selected, detected, diag = _select_and_validate_candidate(
                    model_name,
                    source,
                    cached,
                )
                print(f"{model_name}: validated v2 cache: {selected}")
                candidates = [selected]
                source_mode = "validated_v2_cache"
            except Exception:
                shutil.rmtree(source_cache, ignore_errors=True)
                source_cache.mkdir(parents=True, exist_ok=True)
                candidates = _download_google_drive(source, source_cache)
                source_mode = "google_drive_download"
        else:
            candidates = _download_google_drive(source, source_cache)
            source_mode = "google_drive_download"

    elif is_http_url(source):
        cached = _candidate_files(source_cache)

        if cached:
            try:
                selected, detected, diag = _select_and_validate_candidate(
                    model_name,
                    source,
                    cached,
                )
                print(f"{model_name}: validated v2 cache: {selected}")
                candidates = [selected]
                source_mode = "validated_v2_cache"
            except Exception:
                shutil.rmtree(source_cache, ignore_errors=True)
                source_cache.mkdir(parents=True, exist_ok=True)
                candidates = _download_generic_url(source, source_cache)
                source_mode = "http_download"
        else:
            candidates = _download_generic_url(source, source_cache)
            source_mode = "http_download"

    else:
        raise FileNotFoundError(
            f"{model_name}: source is neither an existing local path nor an HTTP/HTTPS URL: {source}"
        )

    selected, detected, diagnostics = _select_and_validate_candidate(
        model_name,
        source,
        candidates,
    )

    ext = canonical_extension(model_name, detected)
    canonical = source_cache / f"{MODEL_DEFAULT_BASENAMES[model_name]}.{ext}"

    # For local paths, always create an isolated temporary cache copy.
    if selected.resolve() != canonical.resolve():
        shutil.copy2(selected, canonical)

    # Validate the canonical copy again after copying.
    detected2, diagnostics2 = detect_model_format(
        canonical,
        "engine" if ext == "engine" else "auto",
    )

    expected_detected = "torch" if ext in {"pt", "pth"} else ext

    if detected2 != expected_detected:
        raise RuntimeError(
            f"{model_name}: canonical cache copy failed binary verification. "
            f"Expected {expected_detected}, detected {detected2}."
        )

    actual_sha256 = sha256_file(canonical)
    reported_engine_sha256 = KNOWN_STEP4_ENGINE_SHA256.get(model_name)

    metadata = {
        "model": model_name,
        "source": source,
        "source_mode": source_mode,
        "selected_original_name": selected.name,
        "resolved_path": str(canonical),
        "resolved_extension": ext,
        "detected_binary_format": detected2,
        "size_bytes": canonical.stat().st_size,
        "size_mib": canonical.stat().st_size / (1024 ** 2),
        "sha256": actual_sha256,
        "reported_step4_engine_sha256": (
            reported_engine_sha256 if detected2 == "engine" else None
        ),
        "matches_reported_step4_engine_sha256": (
            actual_sha256 == reported_engine_sha256
            if detected2 == "engine" and reported_engine_sha256
            else None
        ),
        "diagnostics": diagnostics2,
    }

    return canonical, metadata


# ---------------------------------------------------------------------
# Resolve ALL three sources first and fail before dataset/tracking if
# even one model source is invalid.
# ---------------------------------------------------------------------

MODEL_PATHS = {}
model_manifest_rows = []
MODEL_SOURCE_ERRORS = {}

for model_name, source in MODEL_SOURCES.items():
    print("\n" + "=" * 96)
    print("MODEL SOURCE PREFLIGHT:", model_name)
    if is_google_drive_url(source):
        print("Drive file ID        :", extract_google_drive_file_id(source))
        print("Drive folder ID      :", extract_google_drive_folder_id(source))
    print("=" * 96)

    try:
        resolved, meta = resolve_model_source(model_name, source)
        MODEL_PATHS[model_name] = resolved
        model_manifest_rows.append(meta)

        print("Resolved path :", resolved)
        print("Detected type :", meta["detected_binary_format"])
        print("Extension     :", meta["resolved_extension"])
        print("Size MiB      :", f"{meta['size_mib']:.2f}")
        print("SHA-256       :", meta["sha256"])

    except Exception as exc:
        MODEL_SOURCE_ERRORS[model_name] = {
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(),
        }
        print("FAILED:", model_name)
        print(exc)


write_json(
    MODEL_META_DIR / "model_source_preflight_errors.json",
    MODEL_SOURCE_ERRORS,
)

if model_manifest_rows:
    MODEL_MANIFEST = pd.DataFrame(model_manifest_rows)
    MODEL_MANIFEST.to_csv(
        MODEL_META_DIR / "model_source_manifest.csv",
        index=False,
    )
    write_json(
        MODEL_META_DIR / "model_source_manifest.json",
        model_manifest_rows,
    )

    display(
        MODEL_MANIFEST[
            [
                "model",
                "selected_original_name",
                "resolved_extension",
                "detected_binary_format",
                "size_mib",
                "sha256",
            ]
        ]
    )

if MODEL_SOURCE_ERRORS:
    raise RuntimeError(
        "MODEL SOURCE PREFLIGHT FAILED. "
        "No detector, tracker, or TrackEval stage will be started. "
        f"Failed models: {list(MODEL_SOURCE_ERRORS)}. "
        f"Inspect {MODEL_META_DIR / 'model_source_preflight_errors.json'} "
        "and the per-model invalid_source_diagnostics JSON files."
    )

print("\n" + "=" * 96)
print("MODEL SOURCE PREFLIGHT: PASSED FOR ALL THREE MODELS")
print("=" * 96)


MODEL SOURCE PREFLIGHT: YOLO26s
Drive file ID        : 10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r
Drive folder ID      : None
Google Drive file ID: 10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r


Downloading...
From (original): https://drive.google.com/uc?id=10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r
From (redirected): https://drive.google.com/uc?id=10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r&confirm=t&uuid=819c4d41-7638-4736-906a-038f318e540b
To: /content/step5_model_cache_v5/YOLO26s/4ab51e0cc1e09828/google_drive_model_download.bin
100%|██████████| 269M/269M [00:01<00:00, 229MB/s]


Downloaded bytes: 268,992,357
Resolved path : /content/step5_model_cache_v5/YOLO26s/4ab51e0cc1e09828/yolo26s_step5_model.engine
Detected type : engine
Extension     : engine
Size MiB      : 256.53
SHA-256       : fc46b73eca3520a3dd29b9d187b8fadcdae23ab215af6347602bb382a63b5dfa

MODEL SOURCE PREFLIGHT: RT-DETR-R18
Drive file ID        : 10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u
Drive folder ID      : None
Google Drive file ID: 10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u


Downloading...
From: https://drive.google.com/uc?id=10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u
To: /content/step5_model_cache_v5/RT-DETR-R18/6ccc85e16300d7b4/google_drive_model_download.bin
100%|██████████| 66.9M/66.9M [00:00<00:00, 200MB/s]


Downloaded bytes: 66,905,348
Resolved path : /content/step5_model_cache_v5/RT-DETR-R18/6ccc85e16300d7b4/rtdetr_r18_step5_model.engine
Detected type : engine
Extension     : engine
Size MiB      : 63.81
SHA-256       : 9d9a2ad34cad42e3b8c80299887552d29729448dda2b278345c3cc3b92257b51

MODEL SOURCE PREFLIGHT: BPD-YOLOn/L-FPN
Drive file ID        : 1cbf-pONCQaaEtzLndOxPxomltRDjmv2a
Drive folder ID      : None
Google Drive file ID: 1cbf-pONCQaaEtzLndOxPxomltRDjmv2a


Downloading...
From (original): https://drive.google.com/uc?id=1cbf-pONCQaaEtzLndOxPxomltRDjmv2a
From (redirected): https://drive.google.com/uc?id=1cbf-pONCQaaEtzLndOxPxomltRDjmv2a&confirm=t&uuid=3cdf9943-0844-43ba-bc60-eb6be0bfb739
To: /content/step5_model_cache_v5/BPD-YOLOn_L-FPN/a3f98ad3094b61a6/google_drive_model_download.bin
100%|██████████| 158M/158M [00:00<00:00, 220MB/s]


Downloaded bytes: 157,862,810
Resolved path : /content/step5_model_cache_v5/BPD-YOLOn_L-FPN/a3f98ad3094b61a6/bpd_yolon_lfpn_step5_model.engine
Detected type : engine
Extension     : engine
Size MiB      : 150.55
SHA-256       : 518d8aafccc491a17e6c219fa12f112dbedfd60b3b0e343004e5e9b3e2e06112


,model,selected_original_name,resolved_extension,detected_binary_format,size_mib,sha256
0,YOLO26s,google_drive_model_download.bin,engine,engine,256.531102,fc46b73eca3520a3dd29b9d187b8fadcdae23ab215af63...
1,RT-DETR-R18,google_drive_model_download.bin,engine,engine,63.805912,9d9a2ad34cad42e3b8c80299887552d29729448dda2b27...
2,BPD-YOLOn/L-FPN,google_drive_model_download.bin,engine,engine,150.549707,518d8aafccc491a17e6c219fa12f112dbedfd60b3b0e34...



MODEL SOURCE PREFLIGHT: PASSED FOR ALL THREE MODELS


## 6. Resolve and extract Okutama-Action — v5 download hotfix

The v4 error `NameError: http_download_robust is not defined` is fixed.

The downloader is now defined before first use and supports:
- Dropbox redirects and direct-download mode;
- retry on transient network failures;
- resume through a `.part` file;
- fresh retry when resume is not supported;
- large multi-gigabyte archives;
- ZIP validation before extraction.

The final L4 configuration still uses the official Okutama `TEST_4K` archive.


In [30]:
def resolve_okutama_source() -> Tuple[Path, Dict[str, Any]]:
    profile = str(DATASET_PROFILE).upper().strip()
    if profile in OKUTAMA_OFFICIAL_URLS:
        source = OKUTAMA_OFFICIAL_URLS[profile]
        archive_name = "okutama_sample_4k.zip" if profile == "SAMPLE_4K" else "okutama_test_4k.zip"
    elif profile == "CUSTOM":
        source = str(OKUTAMA_CUSTOM_SOURCE).strip()
        if not source:
            raise ValueError("DATASET_PROFILE='CUSTOM' but OKUTAMA_CUSTOM_SOURCE is empty.")
        local = Path(os.path.expanduser(source))
        if local.exists() and local.is_dir():
            return local, {"profile": profile, "source": source, "downloaded": False, "archive": None}
        ext = Path(urllib.parse.urlsplit(source).path).suffix
        archive_name = "okutama_custom" + (ext if ext else ".zip")
    else:
        raise ValueError("DATASET_PROFILE must be SAMPLE_4K, TEST_4K, or CUSTOM")

    local_source = Path(os.path.expanduser(source))
    if local_source.exists() and local_source.is_file():
        archive_path = local_source
        downloaded = False
    else:
        if not is_http_url(source):
            raise FileNotFoundError(f"Okutama source is neither a local file/folder nor URL: {source}")
        archive_path = DATA_CACHE / archive_name
        downloaded = True
        if not archive_path.exists() or archive_path.stat().st_size < 1_000_000:
            print("Downloading official/custom Okutama archive:", source)

            http_download_robust(
                source,
                archive_path,
                minimum_bytes=1_000_000,
            )
        else:
            print("Using cached Okutama archive:", archive_path)

    if not zipfile.is_zipfile(archive_path):
        bad_size = (
            archive_path.stat().st_size
            if archive_path.exists()
            else 0
        )

        bad_head = ""

        if archive_path.exists():
            with archive_path.open("rb") as f:
                bad_head = f.read(256).hex()

        raise RuntimeError(
            "Okutama archive exists but is not a valid ZIP file. "
            f"path={archive_path}; "
            f"size_bytes={bad_size}; "
            f"first_bytes_hex={bad_head}. "
            "Delete the bad archive and rerun this cell."
        )

    extract_root = DATA_CACHE / f"extracted_{safe_slug(profile.lower())}"
    marker = extract_root / ".extraction_complete"
    if not marker.exists():
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        print("Testing and extracting:", archive_path)
        with zipfile.ZipFile(archive_path, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise RuntimeError(f"Corrupt ZIP member: {bad}")
            zf.extractall(extract_root)
        marker.write_text(utc_now_iso(), encoding="utf-8")
    else:
        print("Using existing extraction:", extract_root)

    meta = {
        "profile": profile,
        "source": source,
        "downloaded": downloaded,
        "archive": str(archive_path),
        "archive_size_bytes": archive_path.stat().st_size,
        "archive_sha256": sha256_file(archive_path),
        "extract_root": str(extract_root),
    }
    return extract_root, meta


OKUTAMA_ROOT, okutama_source_meta = resolve_okutama_source()
write_json(DATA_META_DIR / "okutama_source.json", okutama_source_meta)
print("Okutama extraction root:", OKUTAMA_ROOT)


Using cached Okutama archive: /content/step5_okutama_cache/okutama_test_4k.zip
Using existing extraction: /content/step5_okutama_cache/extracted_test_4k
Okutama extraction root: /content/step5_okutama_cache/extracted_test_4k


## 7. Parse official labels, pair video/GT and select representative clips
Preferred label directory/file: `SingleActionTrackingLabels`.
Selection is based only on GT density, occlusion and ID turnover—not model results.

In [31]:
def select_seeded_okutama_segments(
    all_candidates: pd.DataFrame,
    max_videos: int,
    seed: int,
    prefer_distinct_scenarios: bool = True,
) -> pd.DataFrame:
    """
    Select one strong candidate segment per video, then choose MAX_VIDEOS
    videos using a reproducible random seed.

    Changing only `seed` changes the selected video set.

    The selection remains scientifically auditable because:
      - one best candidate segment is retained per video;
      - the video list is shuffled reproducibly;
      - distinct scenarios are preferred when available;
      - the selected table is saved with the run outputs.
    """
    if all_candidates.empty:
        raise RuntimeError("No Okutama candidate segments are available.")

    required = {
        "video_key",
        "selection_score",
    }

    missing = required.difference(
        all_candidates.columns
    )

    if missing:
        raise RuntimeError(
            f"Candidate table is missing required columns: {sorted(missing)}"
        )

    # Keep the strongest candidate segment for each video.
    per_video = (
        all_candidates
        .sort_values(
            "selection_score",
            ascending=False,
        )
        .groupby(
            "video_key",
            as_index=False,
            sort=False,
        )
        .head(1)
        .reset_index(drop=True)
    )

    if len(per_video) < max_videos:
        raise RuntimeError(
            f"Requested {max_videos} videos, but only "
            f"{len(per_video)} eligible videos are available."
        )

    # Reproducible shuffle controlled only by VIDEO_SELECTION_SEED.
    shuffled = (
        per_video
        .sample(
            frac=1.0,
            random_state=int(seed),
        )
        .reset_index(drop=True)
    )

    chosen_rows = []
    used_videos = set()
    used_scenarios = set()

    scenario_col = (
        "scenario_key"
        if "scenario_key" in shuffled.columns
        else None
    )

    # First pass: maximize scenario diversity when possible.
    if (
        prefer_distinct_scenarios
        and scenario_col is not None
    ):
        for _, row in shuffled.iterrows():
            if len(chosen_rows) >= int(max_videos):
                break

            video_key = str(
                row["video_key"]
            )

            scenario_key = str(
                row[scenario_col]
            )

            if video_key in used_videos:
                continue

            if scenario_key in used_scenarios:
                continue

            chosen_rows.append(
                row.to_dict()
            )

            used_videos.add(
                video_key
            )

            used_scenarios.add(
                scenario_key
            )

    # Second pass: fill any remaining slots.
    for _, row in shuffled.iterrows():
        if len(chosen_rows) >= int(max_videos):
            break

        video_key = str(
            row["video_key"]
        )

        if video_key in used_videos:
            continue

        chosen_rows.append(
            row.to_dict()
        )

        used_videos.add(
            video_key
        )

    selected_df = pd.DataFrame(
        chosen_rows
    ).reset_index(drop=True)

    if len(selected_df) != int(max_videos):
        raise RuntimeError(
            f"Seeded selection produced {len(selected_df)} videos; "
            f"expected {max_videos}."
        )

    selected_df.insert(
        0,
        "selection_rank",
        np.arange(
            1,
            len(selected_df) + 1,
        ),
    )

    selected_df["video_selection_seed"] = int(
        seed
    )

    return selected_df

def parse_okutama_label_file(path: Path) -> pd.DataFrame:
    rows, errors = [], []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line:
                continue
            try:
                parts = shlex.split(line)
            except Exception as exc:
                errors.append(f"line {line_no}: shlex: {exc}")
                continue
            if len(parts) < 10:
                errors.append(f"line {line_no}: expected >=10 columns, got {len(parts)}")
                continue
            try:
                rows.append({
                    "track_id": int(float(parts[0])),
                    "xmin": float(parts[1]), "ymin": float(parts[2]),
                    "xmax": float(parts[3]), "ymax": float(parts[4]),
                    "frame": int(float(parts[5])),
                    "lost": int(float(parts[6])),
                    "occluded": int(float(parts[7])),
                    "generated": int(float(parts[8])),
                    "label": str(parts[9]).strip('"').strip("'"),
                    "actions": parts[10:], "line_no": line_no,
                })
            except Exception as exc:
                errors.append(f"line {line_no}: parse: {exc}")

    if not rows:
        raise RuntimeError(f"No valid rows parsed from {path}; first errors: {errors[:5]}")
    df = pd.DataFrame(rows)
    df = df[
        (df["label"].str.lower() == "person") & (df["lost"] == 0) &
        (df["xmax"] > df["xmin"]) & (df["ymax"] > df["ymin"])
    ].copy()
    if df.empty:
        raise RuntimeError(f"No valid on-screen person annotations remain in {path}")
    if errors:
        print(f"WARNING: skipped {len(errors)} malformed rows in {path.name}; first: {errors[0]}")
    return df.sort_values(["frame", "track_id"]).reset_index(drop=True)


def extract_video_key(path: Path) -> Optional[str]:
    m = re.search(r"(?<!\d)(\d+\.\d+\.\d+)(?!\d)", str(path))
    if m:
        return m.group(1)
    m = re.search(r"(?<!\d)(\d+[_-]\d+[_-]\d+)(?!\d)", path.stem)
    return re.sub(r"[_-]", ".", m.group(1)) if m else None


def scenario_key(video_key: str) -> str:
    p = video_key.split(".")
    return ".".join(p[1:]) if len(p) == 3 else video_key


def discover_okutama_pairs(root: Path) -> pd.DataFrame:
    video_exts = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}
    videos = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in video_exts)
    text_files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in {".txt", ".csv"})

    tracking_labels = [p for p in text_files if "singleactiontrackinglabels" in str(p).lower().replace("_", "")]
    if tracking_labels:
        label_pool = tracking_labels
        label_mode, consistent_ids = "SingleActionTrackingLabels", True
    else:
        label_pool = [
            p for p in text_files
            if "singleactionlabels" in str(p).lower().replace("_", "")
            and "multi" not in str(p).lower()
        ]
        label_mode, consistent_ids = "SingleActionLabels_fallback_180_frame_IDs", False

    if not videos:
        raise RuntimeError("No videos found. Use SAMPLE_4K/TEST_4K or a CUSTOM video dataset.")
    if not label_pool:
        raise RuntimeError("No SingleActionTrackingLabels or fallback SingleActionLabels found.")

    by_key = defaultdict(list)
    for label in label_pool:
        k = extract_video_key(label)
        if k:
            by_key[k].append(label)

    rows = []
    for video in videos:
        k = extract_video_key(video)
        if not k or not by_key.get(k):
            continue
        candidates = sorted(
            by_key[k],
            key=lambda p: (
                "singleactiontrackinglabels" not in str(p).lower().replace("_", ""),
                len(str(p)),
            ),
        )
        label = candidates[0]
        cap = cv2.VideoCapture(str(video))
        if not cap.isOpened():
            print("WARNING: could not open", video)
            continue
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        rows.append({
            "video_key": k, "scenario_key": scenario_key(k),
            "video_path": str(video), "label_path": str(label),
            "label_mode": label_mode, "consistent_ids": consistent_ids,
            "video_frame_count": frame_count, "fps": fps if fps > 0 else 30.0,
            "width": width, "height": height,
        })

    pairs = pd.DataFrame(rows).drop_duplicates(subset=["video_key", "video_path"]).reset_index(drop=True)
    if pairs.empty:
        raise RuntimeError("Videos/labels exist but could not be paired by official Okutama video key.")
    return pairs


def candidate_segments_for_pair(pair: pd.Series) -> pd.DataFrame:
    labels = parse_okutama_label_file(Path(pair["label_path"]))
    min_frame, max_frame = int(labels["frame"].min()), int(labels["frame"].max())
    frame_base = 0 if min_frame == 0 else 1
    length, stride = int(SEGMENT_LENGTH_FRAMES), int(SEGMENT_STRIDE_FRAMES)
    if not bool(pair["consistent_ids"]):
        length, stride = min(length, 180), 180

    rows = []
    for start in range(min_frame, max_frame - length + 2, max(1, stride)):
        end = start + length - 1
        seg = labels[(labels["frame"] >= start) & (labels["frame"] <= end)].copy()
        if seg.empty:
            continue
        counts = seg.groupby("frame").size().reindex(range(start, end + 1), fill_value=0)
        mean_persons = float(counts.mean())
        if mean_persons < MIN_MEAN_PERSONS_PER_FRAME:
            continue
        unique_ids = int(seg["track_id"].nunique())
        occ = float(seg["occluded"].mean())
        gen = float(seg["generated"].mean())
        h = (seg["ymax"] - seg["ymin"]).clip(lower=0)
        first = seg.groupby("track_id")["frame"].min()
        last = seg.groupby("track_id")["frame"].max()
        turnover = int(((first > start) | (last < end)).sum())
        score = mean_persons + 5.0 * occ + 0.20 * unique_ids + 0.15 * turnover
        rows.append({
            "video_key": pair["video_key"], "scenario_key": pair["scenario_key"],
            "video_path": pair["video_path"], "label_path": pair["label_path"],
            "label_mode": pair["label_mode"], "consistent_ids": bool(pair["consistent_ids"]),
            "frame_base": frame_base, "start_frame": start, "end_frame": end,
            "length": length, "video_start_index": start - frame_base,
            "fps": float(pair["fps"]), "width": int(pair["width"]), "height": int(pair["height"]),
            "mean_persons_per_frame": mean_persons, "unique_gt_ids": unique_ids,
            "occlusion_fraction": occ, "generated_fraction": gen,
            "tiny_fraction_height_lt32": float((h < 32).mean()),
            "very_tiny_fraction_height_lt16": float((h < 16).mean()),
            "id_turnover_count": turnover, "selection_score": score,
        })
    return pd.DataFrame(rows)


OKUTAMA_PAIRS = discover_okutama_pairs(OKUTAMA_ROOT)
candidate_frames = []
for _, pair in tqdm(OKUTAMA_PAIRS.iterrows(), total=len(OKUTAMA_PAIRS), desc="Scanning GT-only candidates"):
    c = candidate_segments_for_pair(pair)
    if not c.empty:
        candidate_frames.append(c)
if not candidate_frames:
    raise RuntimeError("No valid Okutama candidate segments were produced.")
ALL_CANDIDATES = pd.concat(candidate_frames, ignore_index=True)

SELECTED_SEGMENTS = select_seeded_okutama_segments(
    all_candidates=ALL_CANDIDATES,
    max_videos=MAX_VIDEOS,
    seed=VIDEO_SELECTION_SEED,
    prefer_distinct_scenarios=PREFER_DISTINCT_SCENARIOS,
)

selected = SELECTED_SEGMENTS.to_dict("records")

if SELECTED_SEGMENTS.empty:
    raise RuntimeError("Segment selection returned zero clips.")
SELECTED_SEGMENTS["sequence_name"] = [
    safe_slug(f"okutama_{r.video_key}_f{int(r.start_frame):06d}_to_{int(r.end_frame):06d}")
    for r in SELECTED_SEGMENTS.itertuples()
]
OKUTAMA_PAIRS.to_csv(DATA_META_DIR / "discovered_video_label_pairs.csv", index=False)
ALL_CANDIDATES.to_csv(DATA_META_DIR / "all_segment_candidates.csv", index=False)
SELECTED_SEGMENTS.to_csv(DATA_META_DIR / "selected_segments.csv", index=False)

dataset_summary = {
    "number_of_paired_videos": int(len(OKUTAMA_PAIRS)),
    "number_of_candidate_segments": int(len(ALL_CANDIDATES)),
    "number_of_selected_segments": int(len(SELECTED_SEGMENTS)),
    "selected_unique_videos": int(SELECTED_SEGMENTS["video_key"].nunique()),
    "selected_unique_scenarios": int(SELECTED_SEGMENTS["scenario_key"].nunique()),
    "tracking_label_modes": sorted(SELECTED_SEGMENTS["label_mode"].unique().tolist()),
    "consistent_ids": bool(SELECTED_SEGMENTS["consistent_ids"].all()),
}
write_json(DATA_META_DIR / "dataset_summary.json", dataset_summary)
display(SELECTED_SEGMENTS[[
    "sequence_name", "video_key", "start_frame", "end_frame",
    "mean_persons_per_frame", "unique_gt_ids", "occlusion_fraction", "id_turnover_count"
]])


# Save the exact selected videos for scientific audit/reproducibility.
SELECTED_SEGMENTS.to_csv(
    OUTPUT_ROOT / "selected_segments_seeded.csv",
    index=False,
)

write_json(
    OUTPUT_ROOT / "video_selection_config.json",
    {
        "video_selection_seed": int(VIDEO_SELECTION_SEED),
        "max_videos": int(MAX_VIDEOS),
        "selected_video_keys": [
            str(x)
            for x in SELECTED_SEGMENTS["video_key"].tolist()
        ],
    },
)

print(
    f"Selected {len(SELECTED_SEGMENTS)} Okutama videos "
    f"with VIDEO_SELECTION_SEED={VIDEO_SELECTION_SEED}:"
)

display_cols = [
    c
    for c in [
        "selection_rank",
        "video_key",
        "scenario_key",
        "selection_score",
        "video_selection_seed",
    ]
    if c in SELECTED_SEGMENTS.columns
]

display(
    SELECTED_SEGMENTS[
        display_cols
    ]
)


Scanning GT-only candidates:   0%|          | 0/10 [00:00<?, ?it/s]

,sequence_name,video_key,start_frame,end_frame,mean_persons_per_frame,unique_gt_ids,occlusion_fraction,id_turnover_count
0,okutama_2.2.1_f000000_to_000299,2.2.1,0,299,4.933333,7,0.000000,6
1,okutama_1.1.9_f000300_to_000599,1.1.9,300,599,7.300000,8,0.009132,2
2,okutama_1.2.10_f000300_to_000599,1.2.10,300,599,7.760000,16,0.033935,16
3,okutama_2.1.8_f000300_to_000599,2.1.8,300,599,4.733333,8,0.119718,8
4,okutama_1.2.3_f000300_to_000599,1.2.3,300,599,5.533333,15,0.012048,13


Selected 5 Okutama videos with VIDEO_SELECTION_SEED=42:


,selection_rank,video_key,scenario_key,selection_score,video_selection_seed
0,1,2.2.1,2.1,7.233333,42
1,2,1.1.9,1.9,9.245662,42
2,3,1.2.10,2.10,13.529674,42
3,4,2.1.8,1.8,8.131925,42
4,5,1.2.3,2.3,10.543574,42


## 8. BPD-YOLOn/L-FPN DySample compatibility

In [32]:
import torch.nn as nn
import torch.nn.functional as F


class DySample(nn.Module):
    """Project-compatible content-aware point-sampling upsampler."""
    def __init__(self, channels: Optional[int] = None, scale: int = 2, max_offset: float = 0.25):
        super().__init__()
        self.channels = int(channels) if channels is not None else None
        self.scale = int(scale)
        self.max_offset = float(max_offset)
        self._zero_initialized = False
        if self.channels is None:
            self.offset = nn.LazyConv2d(2, kernel_size=1, stride=1, padding=0)
        else:
            self.offset = nn.Conv2d(self.channels, 2, kernel_size=1, stride=1, padding=0)
            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)
                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)
            self._zero_initialized = True

    def _materialize_legacy_lazy_offset(self, x):
        if isinstance(self.offset, nn.LazyConv2d) and not getattr(self, "_zero_initialized", False):
            _ = self.offset(x)
            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)
                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)
            self._zero_initialized = True

    def forward(self, x):
        channels = getattr(self, "channels", None)
        if channels is not None and x.shape[1] != int(channels):
            raise RuntimeError(f"DySample expected {int(channels)} channels but received {x.shape[1]}")
        self._materialize_legacy_lazy_offset(x)
        b, _, h, w = x.shape
        scale = int(getattr(self, "scale", 2))
        max_offset = float(getattr(self, "max_offset", 0.25))
        oh, ow = h * scale, w * scale
        raw = self.offset(x)
        raw = F.interpolate(raw, size=(oh, ow), mode="bilinear", align_corners=False)
        raw = torch.tanh(raw) * max_offset
        ys = ((torch.arange(oh, device=x.device, dtype=x.dtype) + 0.5) / oh) * 2.0 - 1.0
        xs = ((torch.arange(ow, device=x.device, dtype=x.dtype) + 0.5) / ow) * 2.0 - 1.0
        yy, xx = torch.meshgrid(ys, xs, indexing="ij")
        base = torch.stack((xx, yy), dim=-1).unsqueeze(0).expand(b, -1, -1, -1)
        dx = raw[:, 0] * (2.0 / max(w, 1))
        dy = raw[:, 1] * (2.0 / max(h, 1))
        grid = base + torch.stack((dx, dy), dim=-1)
        return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)


def register_bpd_compatibility():
    import ultralytics.nn.tasks as ultralytics_tasks
    setattr(sys.modules["__main__"], "DySample", DySample)
    setattr(ultralytics_tasks, "DySample", DySample)
    try:
        import ultralytics.nn.modules as ultralytics_modules
        setattr(ultralytics_modules, "DySample", DySample)
    except Exception:
        pass
    try:
        torch.serialization.add_safe_globals([DySample])
    except Exception:
        pass


register_bpd_compatibility()
print("BPD DySample compatibility: REGISTERED")


BPD DySample compatibility: REGISTERED


## 9. Pinned official RT-DETR source
Required for the final `.pth` checkpoint. If the Step-4 `.onnx` export is supplied,
ONNX Runtime is used directly instead.

In [33]:
RTDETR_LOCAL_ROOT = TOOL_CACHE / "rtdetr"
RTDETR_REPO_ROOT = RTDETR_LOCAL_ROOT / "RT-DETR"
RTDETR_ROOT = RTDETR_REPO_ROOT / "rtdetrv2_pytorch"
RTDETR_CONFIG_DIR = RTDETR_ROOT / "configs" / "custom"
RTDETR_CONFIG_PATH = RTDETR_CONFIG_DIR / "rtdetr_r18_person_step5.yml"


def prepare_rtdetr_repository() -> None:
    if not RTDETR_REPO_ROOT.exists():
        RTDETR_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["git", "clone", "--filter=blob:none", RTDETR_REPO_URL, str(RTDETR_REPO_ROOT)])
    subprocess.check_call(["git", "-C", str(RTDETR_REPO_ROOT), "fetch", "--all", "--tags", "--prune"])
    subprocess.check_call(["git", "-C", str(RTDETR_REPO_ROOT), "checkout", "--force", RTDETR_REPO_COMMIT])
    if not RTDETR_ROOT.exists():
        raise FileNotFoundError(RTDETR_ROOT)
    RTDETR_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    config_text = f'''__include__:
  - ../rtdetr/rtdetr_r18vd_6x_coco.yml

num_classes: 1
remap_mscoco_category: False

eval_spatial_size: [{IMAGE_SIZE}, {IMAGE_SIZE}]

PResNet:
  pretrained: False
'''
    RTDETR_CONFIG_PATH.write_text(config_text, encoding="utf-8")
    commit = subprocess.check_output(["git", "-C", str(RTDETR_REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
    write_json(AUDIT_DIR / "rtdetr_source.json", {
        "repo": RTDETR_REPO_URL, "requested_commit": RTDETR_REPO_COMMIT,
        "resolved_commit": commit, "config": str(RTDETR_CONFIG_PATH),
    })


if MODEL_PATHS["RT-DETR-R18"].suffix.lower() == ".pth":
    prepare_rtdetr_repository()
    print("RT-DETR repository/config: READY")
else:
    print("RT-DETR ONNX source selected; PyTorch repository setup deferred/not needed.")


RT-DETR ONNX source selected; PyTorch repository setup deferred/not needed.


## 10. Unified detector backends

In [34]:
from ultralytics import YOLO


@dataclass
class DetectionBatch:
    boxes: np.ndarray
    scores: np.ndarray
    classes: np.ndarray
    preprocess_ms: float = 0.0
    inference_ms: float = 0.0
    postprocess_ms: float = 0.0
    total_ms: float = 0.0

    def __len__(self):
        return int(len(self.boxes))

    @staticmethod
    def empty(total_ms: float = 0.0):
        return DetectionBatch(
            np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32),
            np.zeros((0,), dtype=np.float32), total_ms=float(total_ms)
        )


class UltralyticsDetector:
    def __init__(self, model_path: Path, model_name: str, is_bpd: bool = False):
        self.model_path = Path(model_path)
        self.model_name = model_name
        if is_bpd and self.model_path.suffix.lower() == ".pt":
            register_bpd_compatibility()
        self.model = YOLO(str(self.model_path))
        try:
            self.model.names = {0: "person"}
        except Exception:
            pass

    def predict(self, frame: np.ndarray) -> DetectionBatch:
        cuda_sync(); total_start = time.perf_counter()
        results = self.model.predict(
            source=frame, imgsz=IMAGE_SIZE, conf=detector_conf_threshold(self.model_name),
            iou=DETECTOR_NMS_IOU, max_det=MAX_DETECTIONS, classes=[0],
            device=DEVICE_ID,
            verbose=False,
        )
        cuda_sync(); total_ms = (time.perf_counter() - total_start) * 1000.0
        if not results:
            return DetectionBatch.empty(total_ms)
        result = results[0]
        speed = getattr(result, "speed", {}) or {}
        if result.boxes is None or len(result.boxes) == 0:
            return DetectionBatch(
                np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.float32),
                float(speed.get("preprocess", 0.0) or 0.0), float(speed.get("inference", 0.0) or 0.0),
                float(speed.get("postprocess", 0.0) or 0.0), total_ms
            )
        boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
        scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
        mask = scores >= detector_conf_threshold(self.model_name)
        boxes, scores = boxes[mask], scores[mask]
        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]; boxes, scores = boxes[order], scores[order]
        return DetectionBatch(
            boxes, scores, np.zeros((len(boxes),), np.float32),
            float(speed.get("preprocess", 0.0) or 0.0), float(speed.get("inference", 0.0) or 0.0),
            float(speed.get("postprocess", 0.0) or 0.0), total_ms
        )


class RTDETRPyTorchDetector:
    def __init__(self, checkpoint_path: Path):
        self.checkpoint_path = Path(checkpoint_path)
        self._build()

    @staticmethod
    def _extract_state_dict(checkpoint):
        if not isinstance(checkpoint, dict):
            raise RuntimeError("Unexpected RT-DETR checkpoint format")
        ema = checkpoint.get("ema")
        if ema is not None:
            if isinstance(ema, dict) and "module" in ema and isinstance(ema["module"], dict):
                return ema["module"]
            if isinstance(ema, dict):
                return ema
            if hasattr(ema, "state_dict"):
                return ema.state_dict()
        model_state = checkpoint.get("model")
        if model_state is not None:
            if isinstance(model_state, dict):
                return model_state
            if hasattr(model_state, "state_dict"):
                return model_state.state_dict()
        raise RuntimeError("Could not find RT-DETR weights under 'ema' or 'model'")

    def _build(self):
        prepare_rtdetr_repository()
        if str(RTDETR_ROOT) not in sys.path:
            sys.path.insert(0, str(RTDETR_ROOT))
        importlib.invalidate_caches()
        old_cwd = Path.cwd()
        try:
            os.chdir(RTDETR_ROOT)
            from src.core import YAMLConfig
            cfg = YAMLConfig(str(RTDETR_CONFIG_PATH), device=f"cuda:{DEVICE_ID}", use_amp=False)
        finally:
            os.chdir(old_cwd)
        checkpoint = torch.load(self.checkpoint_path, map_location="cpu", weights_only=False)
        state = self._extract_state_dict(checkpoint)
        cfg.model.load_state_dict(state, strict=True)
        self.model = cfg.model.deploy().cuda(DEVICE_ID).eval()
        self.postprocessor = cfg.postprocessor.deploy()
        del checkpoint, state
        gc.collect(); torch.cuda.empty_cache()

    def _preprocess(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
        array = resized.astype(np.float32) / 255.0
        return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).contiguous().cuda(DEVICE_ID, non_blocking=True)

    def predict(self, frame: np.ndarray) -> DetectionBatch:
        h, w = frame.shape[:2]
        total_start = time.perf_counter()
        pre_start = time.perf_counter()
        tensor = self._preprocess(frame)
        original_size = torch.tensor([[w, h]], dtype=torch.float32, device=f"cuda:{DEVICE_ID}")
        cuda_sync(); preprocess_ms = (time.perf_counter() - pre_start) * 1000.0
        cuda_sync(); infer_start = time.perf_counter()
        with torch.inference_mode():
            ctx = torch.autocast("cuda", dtype=torch.float16) if USE_FP16_CHECKPOINT_INFERENCE else contextlib.nullcontext()
            with ctx:
                outputs = self.model(tensor)
        cuda_sync(); inference_ms = (time.perf_counter() - infer_start) * 1000.0
        post_start = time.perf_counter()
        with torch.inference_mode():
            labels, boxes, scores = self.postprocessor(outputs, original_size)
        cuda_sync()
        labels = labels[0].detach().cpu().numpy()
        boxes = boxes[0].detach().cpu().numpy().astype(np.float32)
        scores = scores[0].detach().cpu().numpy().astype(np.float32)
        mask = (scores >= detector_conf_threshold("RT-DETR-R18")) & (labels.astype(np.int64) == 0)
        boxes, scores = boxes[mask], scores[mask]
        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]; boxes, scores = boxes[order], scores[order]
        postprocess_ms = (time.perf_counter() - post_start) * 1000.0
        total_ms = (time.perf_counter() - total_start) * 1000.0
        return DetectionBatch(boxes, scores, np.zeros((len(boxes),), np.float32),
                              preprocess_ms, inference_ms, postprocess_ms, total_ms)


def ort_numpy_dtype(input_type: str):
    t = str(input_type).lower()
    if "int64" in t: return np.int64
    if "int32" in t: return np.int32
    if "float16" in t: return np.float16
    return np.float32


class RTDETRONNXDetector:
    def __init__(self, onnx_path: Path):
        import onnxruntime as ort
        available = ort.get_available_providers()
        providers = (["CUDAExecutionProvider"] if "CUDAExecutionProvider" in available else []) + ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.inputs = {x.name: x for x in self.session.get_inputs()}
        self.output_names = [x.name for x in self.session.get_outputs()]
        if not {"images", "orig_target_sizes"}.issubset(self.inputs):
            raise RuntimeError(f"Unexpected RT-DETR ONNX inputs: {list(self.inputs)}")
        if not {"labels", "boxes", "scores"}.issubset(self.output_names):
            raise RuntimeError(f"Unexpected RT-DETR ONNX outputs: {self.output_names}")
        print("RT-DETR ONNX providers:", self.session.get_providers())

    def predict(self, frame: np.ndarray) -> DetectionBatch:
        h, w = frame.shape[:2]; total_start = time.perf_counter(); pre = time.perf_counter()
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
        images = (resized.astype(np.float32).transpose(2, 0, 1)[None] / 255.0).astype(ort_numpy_dtype(self.inputs["images"].type), copy=False)
        sizes = np.asarray([[w, h]], dtype=ort_numpy_dtype(self.inputs["orig_target_sizes"].type))
        preprocess_ms = (time.perf_counter() - pre) * 1000.0
        inf = time.perf_counter(); outputs = self.session.run(self.output_names, {"images": images, "orig_target_sizes": sizes})
        inference_ms = (time.perf_counter() - inf) * 1000.0; post = time.perf_counter()
        out = dict(zip(self.output_names, outputs))
        labels, boxes, scores = np.asarray(out["labels"])[0], np.asarray(out["boxes"])[0].astype(np.float32), np.asarray(out["scores"])[0].astype(np.float32)
        mask = (scores >= detector_conf_threshold("RT-DETR-R18")) & (labels.astype(np.int64) == 0)
        boxes, scores = boxes[mask], scores[mask]
        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]; boxes, scores = boxes[order], scores[order]
        postprocess_ms = (time.perf_counter() - post) * 1000.0
        total_ms = (time.perf_counter() - total_start) * 1000.0
        return DetectionBatch(boxes, scores, np.zeros((len(boxes),), np.float32), preprocess_ms, inference_ms, postprocess_ms, total_ms)



def trt_dtype_to_torch(dtype):
    np_dtype = np.dtype(trt.nptype(dtype))

    mapping = {
        np.dtype(np.float32): torch.float32,
        np.dtype(np.float16): torch.float16,
        np.dtype(np.int64): torch.int64,
        np.dtype(np.int32): torch.int32,
        np.dtype(np.int8): torch.int8,
        np.dtype(np.bool_): torch.bool,
    }

    if np_dtype not in mapping:
        raise TypeError(f"Unsupported TensorRT dtype: {dtype} / {np_dtype}")

    return mapping[np_dtype]


class RTDETRTensorRTDetector:
    def __init__(self, engine_path: Path):
        self.engine_path = Path(engine_path)
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)

        engine_bytes = self.engine_path.read_bytes()
        self.engine = self.runtime.deserialize_cuda_engine(engine_bytes)

        if self.engine is None:
            raise RuntimeError(
                "Could not deserialize the RT-DETR TensorRT engine. "
                "TensorRT engines are hardware/runtime specific. "
                f"Engine={self.engine_path}, TensorRT={trt.__version__}, "
                f"GPU={torch.cuda.get_device_name(DEVICE_ID)}"
            )

        self.context = self.engine.create_execution_context()

        if self.context is None:
            raise RuntimeError("Could not create RT-DETR TensorRT execution context.")

        self.stream = torch.cuda.Stream(device=DEVICE_ID)
        self.input_names = []
        self.output_names = []

        for index in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(index)
            mode = self.engine.get_tensor_mode(name)

            if mode == trt.TensorIOMode.INPUT:
                self.input_names.append(name)
            else:
                self.output_names.append(name)

        if not {"images", "orig_target_sizes"}.issubset(self.input_names):
            raise RuntimeError(f"Unexpected RT-DETR TensorRT inputs: {self.input_names}")

        if not {"labels", "boxes", "scores"}.issubset(self.output_names):
            raise RuntimeError(f"Unexpected RT-DETR TensorRT outputs: {self.output_names}")

        self.buffers = {}
        self._initialize_buffers()

        print("RT-DETR TensorRT engine: LOADED")
        print("Inputs :", self.input_names)
        print("Outputs:", self.output_names)

    def _initialize_buffers(self):
        input_shapes = {
            "images": (1, 3, IMAGE_SIZE, IMAGE_SIZE),
            "orig_target_sizes": (1, 2),
        }

        for name in self.input_names:
            engine_shape = tuple(
                int(v) for v in self.engine.get_tensor_shape(name)
            )

            if any(v < 0 for v in engine_shape):
                shape = input_shapes.get(name)

                if shape is None:
                    raise RuntimeError(f"No static input-shape rule for: {name}")

                ok = self.context.set_input_shape(name, shape)

                if ok is False:
                    raise RuntimeError(f"TensorRT rejected {name} shape {shape}")

        for index in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(index)
            dtype = self.engine.get_tensor_dtype(name)
            torch_dtype = trt_dtype_to_torch(dtype)
            shape = tuple(
                int(v) for v in self.context.get_tensor_shape(name)
            )

            if any(v < 0 for v in shape):
                raise RuntimeError(
                    f"TensorRT tensor still has dynamic shape: {name} -> {shape}"
                )

            tensor = torch.empty(
                shape,
                dtype=torch_dtype,
                device=f"cuda:{DEVICE_ID}",
            )

            self.buffers[name] = tensor
            self.context.set_tensor_address(
                name,
                int(tensor.data_ptr()),
            )

    def _prepare_inputs(self, frame):
        height, width = frame.shape[:2]

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(
            rgb,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_LINEAR,
        )

        image_np = (
            resized.astype(np.float32)
            .transpose(2, 0, 1)[None]
            / 255.0
        )

        image_tensor = torch.from_numpy(image_np).to(
            device=f"cuda:{DEVICE_ID}",
            dtype=self.buffers["images"].dtype,
        )

        size_tensor = torch.tensor(
            [[width, height]],
            device=f"cuda:{DEVICE_ID}",
            dtype=self.buffers["orig_target_sizes"].dtype,
        )

        self.buffers["images"].copy_(image_tensor)
        self.buffers["orig_target_sizes"].copy_(size_tensor)

    def predict(self, frame):
        total_start = time.perf_counter()

        pre_start = time.perf_counter()
        self._prepare_inputs(frame)
        cuda_sync()
        preprocess_ms = (time.perf_counter() - pre_start) * 1000.0

        cuda_sync()
        infer_start = time.perf_counter()

        ok = self.context.execute_async_v3(
            stream_handle=int(self.stream.cuda_stream)
        )

        if not ok:
            raise RuntimeError("RT-DETR TensorRT execute_async_v3 returned False.")

        self.stream.synchronize()
        inference_ms = (time.perf_counter() - infer_start) * 1000.0

        post_start = time.perf_counter()

        labels = self.buffers["labels"][0].detach().cpu().numpy()
        boxes = (
            self.buffers["boxes"][0]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        scores = (
            self.buffers["scores"][0]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        mask = (
            (scores >= detector_conf_threshold("RT-DETR-R18"))
            & (labels.astype(np.int64) == 0)
        )

        boxes = boxes[mask]
        scores = scores[mask]

        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]
            boxes = boxes[order]
            scores = scores[order]

        postprocess_ms = (time.perf_counter() - post_start) * 1000.0
        total_ms = (time.perf_counter() - total_start) * 1000.0

        return DetectionBatch(
            boxes=boxes,
            scores=scores,
            classes=np.zeros((len(boxes),), dtype=np.float32),
            preprocess_ms=preprocess_ms,
            inference_ms=inference_ms,
            postprocess_ms=postprocess_ms,
            total_ms=total_ms,
        )


def build_detector(model_name: str):
    path = MODEL_PATHS[model_name]
    suffix = path.suffix.lower()

    if model_name == "RT-DETR-R18":
        if suffix == ".pth":
            return RTDETRPyTorchDetector(path)
        if suffix == ".onnx":
            return RTDETRONNXDetector(path)
        if suffix == ".engine":
            return RTDETRTensorRTDetector(path)
        raise ValueError("RT-DETR Step-5 supports .pth, .onnx, or .engine.")

    if model_name == "YOLO26s":
        if suffix not in {".pt", ".onnx", ".engine"}:
            raise ValueError(f"Unsupported YOLO26s format: {suffix}")
        return UltralyticsDetector(path, model_name, False)

    if model_name == "BPD-YOLOn/L-FPN":
        if suffix not in {".pt", ".onnx", ".engine"}:
            raise ValueError(f"Unsupported BPD format: {suffix}")
        return UltralyticsDetector(path, model_name, suffix == ".pt")

    raise KeyError(model_name)


print("Unified detector backends: READY")


Unified detector backends: READY


## 11. Common ByteTrack / BoT-SORT adapters
Both receive the exact same detector boxes/scores. BoT-SORT uses sparse-optical-flow
camera-motion compensation; ReID is disabled in the primary controlled comparison.

In [35]:
from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.trackers.bot_sort import BOTSORT


@dataclass
class TrackBatch:
    boxes: np.ndarray
    ids: np.ndarray
    scores: np.ndarray
    def __len__(self): return int(len(self.ids))
    @staticmethod
    def empty():
        return TrackBatch(np.zeros((0, 4), np.float32), np.zeros((0,), np.int32), np.zeros((0,), np.float32))


class CommonTracker:
    """
    Unified ByteTrack / BoT-SORT adapter with an ID-stable configuration.

    The tracker keeps lost identities alive for TRACK_LOST_GRACE_SECONDS.
    For Ultralytics 8.4.116, track_buffer is a frame count, therefore it is
    calculated from the real FPS of each video.
    """

    def __init__(
        self,
        model_name: str,
        tracker_name: str,
        frame_rate: float,
    ):
        self.model_name = str(model_name)
        self.tracker_name = str(tracker_name)
        self.frame_rate = max(
            1.0,
            float(frame_rate),
        )

        self.track_buffer_frames = max(
            1,
            int(
                round(
                    self.frame_rate
                    * TRACK_LOST_GRACE_SECONDS
                )
            ),
        )

        self.effective_settings = None

        self._build()

    def _settings(self) -> Dict[str, Any]:
        if self.model_name not in TRACKER_MODEL_THRESHOLDS:
            raise KeyError(
                f"No tracker threshold profile for detector: {self.model_name}"
            )

        if self.tracker_name == "ByteTrack":
            settings = dict(
                BYTETRACK_SETTINGS
            )

        elif self.tracker_name == "BoT-SORT":
            settings = dict(
                BOTSORT_SETTINGS
            )

        else:
            raise ValueError(
                f"Unsupported tracker: {self.tracker_name}"
            )

        settings.update(
            TRACKER_MODEL_THRESHOLDS[
                self.model_name
            ]
        )

        # Exact 3-second lost-track retention based on source-video FPS.
        settings["track_buffer"] = int(
            self.track_buffer_frames
        )

        return settings

    def _build(self):
        settings = self._settings()

        if self.tracker_name == "ByteTrack":
            tracker_cls = BYTETracker
            tracker_type = "bytetrack"

        elif self.tracker_name == "BoT-SORT":
            tracker_cls = BOTSORT
            tracker_type = "botsort"

        else:
            raise ValueError(
                f"Unsupported tracker: {self.tracker_name}"
            )

        args = SimpleNamespace(
            tracker_type=tracker_type,
            **settings,
        )

        self.effective_settings = {
            **settings,
            "tracker_type": tracker_type,
            "model_name": self.model_name,
            "frame_rate": self.frame_rate,
            "track_lost_grace_seconds": (
                self.track_buffer_frames
                / self.frame_rate
            ),
        }

        # Ultralytics 8.4.116 constructors take args only.
        # The fallback keeps this adapter compatible with nearby versions.
        try:
            self.tracker = tracker_cls(
                args,
                frame_rate=self.frame_rate,
            )

        except TypeError:
            self.tracker = tracker_cls(
                args
            )

        # Verify the actual buffer used by the tracker implementation.
        actual_buffer = getattr(
            self.tracker,
            "max_frames_lost",
            getattr(
                self.tracker,
                "max_time_lost",
                self.track_buffer_frames,
            ),
        )

        self.effective_settings[
            "actual_tracker_buffer_frames"
        ] = int(
            actual_buffer
        )

        self.effective_settings[
            "actual_tracker_buffer_seconds"
        ] = float(
            actual_buffer
            / self.frame_rate
        )

        self.effective_settings[
            "internal_reid_active"
        ] = bool(
            self.tracker_name == "BoT-SORT"
            and settings.get("with_reid", False)
        )

    def reset(self):
        self._build()

    def audit_dict(self) -> Dict[str, Any]:
        return dict(
            self.effective_settings
            or {}
        )

    def update(
        self,
        detections: DetectionBatch,
        frame: np.ndarray,
    ) -> TrackBatch:
        h, w = frame.shape[:2]

        if len(detections) == 0:
            data = torch.empty(
                (0, 6),
                dtype=torch.float32,
            )

        else:
            cls_col = np.zeros(
                (len(detections), 1),
                np.float32,
            )

            data = torch.from_numpy(
                np.concatenate(
                    [
                        detections.boxes.astype(
                            np.float32
                        ),
                        detections.scores[
                            :,
                            None,
                        ].astype(
                            np.float32
                        ),
                        cls_col,
                    ],
                    axis=1,
                )
            )

        results = Boxes(
            data,
            (h, w),
        )

        tracks = self.tracker.update(
            results,
            frame,
        )

        if (
            tracks is None
            or len(tracks) == 0
        ):
            return TrackBatch.empty()

        tracks = np.asarray(
            tracks,
            dtype=np.float32,
        )

        if tracks.ndim == 1:
            tracks = tracks[
                None,
                :
            ]

        if tracks.shape[1] < 6:
            raise RuntimeError(
                f"Unexpected {self.tracker_name} "
                f"output shape: {tracks.shape}"
            )

        return TrackBatch(
            tracks[
                :,
                :4,
            ].astype(
                np.float32
            ),
            tracks[
                :,
                4,
            ].astype(
                np.int32
            ),
            tracks[
                :,
                5,
            ].astype(
                np.float32
            ),
        )

print("Tracker adapters: READY")
print(
    "ID-stable lost-track grace:",
    f"{TRACK_LOST_GRACE_SECONDS:.1f} seconds",
)
print(
    "BoT-SORT ReID:",
    BOTSORT_SETTINGS["with_reid"],
    "| model:",
    BOTSORT_SETTINGS["model"],
)


Tracker adapters: READY
ID-stable lost-track grace: 3.0 seconds
BoT-SORT ReID: True | model: yolo26n-reid.onnx


## 11B. Long-Term ReID Global-ID memory

The local tracker ID is not assumed to be permanent. When a subject disappears completely and later
returns with a new local ID, the following layer compares the new appearance embedding against
inactive identities stored in a time-limited gallery.

The formal MOT output uses **Global IDs**. Raw Local-ID MOT output is saved beside it for audit.


In [36]:
from ultralytics.trackers.utils.reid import ReID


def _l2_normalize(vector: Optional[np.ndarray]) -> Optional[np.ndarray]:
    if vector is None:
        return None
    arr = np.asarray(vector, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(arr))
    if norm < 1e-12:
        return None
    return arr / norm


def _cosine_similarity(
    a: Optional[np.ndarray],
    b: Optional[np.ndarray],
) -> float:
    a = _l2_normalize(a)
    b = _l2_normalize(b)

    if a is None or b is None:
        return float("nan")

    return float(
        np.clip(
            np.dot(a, b),
            -1.0,
            1.0,
        )
    )


def _xyxy_to_center_xywh(
    boxes: np.ndarray,
) -> np.ndarray:
    boxes = np.asarray(
        boxes,
        dtype=np.float32,
    )

    if boxes.size == 0:
        return np.zeros(
            (0, 4),
            dtype=np.float32,
        )

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    return np.stack(
        [
            (x1 + x2) / 2.0,
            (y1 + y2) / 2.0,
            np.maximum(1.0, x2 - x1),
            np.maximum(1.0, y2 - y1),
        ],
        axis=1,
    ).astype(
        np.float32
    )


def _crop_hsv_histogram(
    frame: np.ndarray,
    box: np.ndarray,
) -> Optional[np.ndarray]:
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = [
        int(round(v))
        for v in box
    ]

    x1 = max(
        0,
        min(w - 1, x1),
    )
    y1 = max(
        0,
        min(h - 1, y1),
    )
    x2 = max(
        x1 + 1,
        min(w, x2),
    )
    y2 = max(
        y1 + 1,
        min(h, y2),
    )

    crop = frame[
        y1:y2,
        x1:x2,
    ]

    if crop.size == 0:
        return None

    hsv = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2HSV,
    )

    hist = cv2.calcHist(
        [hsv],
        [0, 1],
        None,
        [16, 8],
        [0, 180, 0, 256],
    )

    hist = hist.astype(
        np.float32
    ).reshape(-1)

    norm = float(
        np.linalg.norm(hist)
    )

    if norm < 1e-12:
        return None

    return hist / norm


def _histogram_similarity(
    a: Optional[np.ndarray],
    b: Optional[np.ndarray],
) -> float:
    if a is None or b is None:
        return 0.5

    corr = float(
        cv2.compareHist(
            np.asarray(a, np.float32),
            np.asarray(b, np.float32),
            cv2.HISTCMP_CORREL,
        )
    )

    return float(
        np.clip(
            (corr + 1.0) / 2.0,
            0.0,
            1.0,
        )
    )


def _box_area(
    box: np.ndarray,
) -> float:
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    return max(
        1.0,
        (x2 - x1)
        * (y2 - y1),
    )


def _size_similarity(
    a: np.ndarray,
    b: np.ndarray,
) -> float:
    aa = _box_area(a)
    bb = _box_area(b)

    return float(
        min(aa, bb)
        / max(aa, bb)
    )


def _border_side(
    box: np.ndarray,
    frame_width: int,
    frame_height: int,
) -> str:
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    cx = (
        x1 + x2
    ) / 2.0
    cy = (
        y1 + y2
    ) / 2.0

    mx = (
        float(frame_width)
        * GLOBAL_REID_BORDER_MARGIN_RATIO
    )

    my = (
        float(frame_height)
        * GLOBAL_REID_BORDER_MARGIN_RATIO
    )

    candidates = []

    if cx <= mx:
        candidates.append(
            (
                cx / max(mx, 1.0),
                "left",
            )
        )

    if cx >= frame_width - mx:
        candidates.append(
            (
                (frame_width - cx)
                / max(mx, 1.0),
                "right",
            )
        )

    if cy <= my:
        candidates.append(
            (
                cy / max(my, 1.0),
                "top",
            )
        )

    if cy >= frame_height - my:
        candidates.append(
            (
                (frame_height - cy)
                / max(my, 1.0),
                "bottom",
            )
        )

    if not candidates:
        return "center"

    candidates.sort(
        key=lambda item: item[0]
    )

    return str(
        candidates[0][1]
    )


@dataclass
class GlobalIdentityRecord:
    global_id: int
    embedding: Optional[np.ndarray]
    histogram: Optional[np.ndarray]
    last_box: np.ndarray
    last_seen_frame: int
    last_local_id: int
    active_local_id: Optional[int]
    last_border_side: str
    seen_updates: int = 0


_GLOBAL_REID_ENCODER = None


def get_global_reid_encoder():
    global _GLOBAL_REID_ENCODER

    if (
        not ENABLE_GLOBAL_ID_RECOVERY
    ):
        return None

    if _GLOBAL_REID_ENCODER is not None:
        return _GLOBAL_REID_ENCODER

    print(
        "Loading Long-Term ReID encoder:",
        GLOBAL_REID_MODEL,
    )

    try:
        _GLOBAL_REID_ENCODER = ReID(
            GLOBAL_REID_MODEL,
            device=f"cuda:{DEVICE_ID}",
            fp16=True,
        )

    except Exception:
        if GLOBAL_REID_REQUIRED:
            raise

        print(
            "WARNING: Global ReID encoder could not be loaded. "
            "Global-ID recovery is disabled for this run."
        )

        _GLOBAL_REID_ENCODER = False

    return (
        None
        if _GLOBAL_REID_ENCODER is False
        else _GLOBAL_REID_ENCODER
    )


class GlobalIdentityManager:
    """
    Conservative long-term identity memory.

    Local tracker IDs are mapped to stable Global IDs. A new local track may
    recover a previous Global ID only when:
      1) the old identity is currently inactive,
      2) it is still inside the gallery time window,
      3) cosine appearance similarity is high enough,
      4) the weighted score passes the global gate, and
      5) the best candidate is clearly better than the runner-up.
    """

    def __init__(
        self,
        frame_rate: float,
        frame_width: int,
        frame_height: int,
    ):
        self.frame_rate = max(
            1.0,
            float(frame_rate),
        )

        self.frame_width = int(
            frame_width
        )
        self.frame_height = int(
            frame_height
        )

        self.memory_frames = max(
            1,
            int(
                round(
                    GLOBAL_ID_MEMORY_SECONDS
                    * self.frame_rate
                )
            ),
        )

        self.encoder = (
            get_global_reid_encoder()
            if ENABLE_GLOBAL_ID_RECOVERY
            else None
        )

        self.records: Dict[
            int,
            GlobalIdentityRecord,
        ] = {}

        self.local_to_global: Dict[
            int,
            int,
        ] = {}

        self.next_global_id = 1

        self.unique_local_ids = set()
        self.unique_global_ids = set()

        self.recovery_events: List[
            Dict[str, Any]
        ] = []

        self.new_identity_events = 0
        self.recovered_identity_events = 0
        self.ambiguous_rejections = 0
        self.no_feature_rejections = 0

    def _geometry_is_valid(
        self,
        box: np.ndarray,
    ) -> bool:
        x1, y1, x2, y2 = map(
            float,
            box,
        )

        return bool(
            (x2 - x1)
            >= GLOBAL_REID_MIN_BOX_WIDTH_PX
            and
            (y2 - y1)
            >= GLOBAL_REID_MIN_BOX_HEIGHT_PX
        )

    def _extract_features(
        self,
        frame: np.ndarray,
        boxes: np.ndarray,
        indices: Sequence[int],
    ) -> Dict[
        int,
        Optional[np.ndarray],
    ]:
        indices = [
            int(i)
            for i in indices
        ]

        out = {
            i: None
            for i in indices
        }

        if (
            not indices
            or self.encoder is None
        ):
            return out

        valid_indices = [
            i
            for i in indices
            if self._geometry_is_valid(
                boxes[i]
            )
        ]

        if not valid_indices:
            return out

        xywh = _xyxy_to_center_xywh(
            boxes[
                valid_indices
            ]
        )

        features = self.encoder(
            frame,
            xywh,
        )

        for idx, feature in zip(
            valid_indices,
            features,
        ):
            out[idx] = _l2_normalize(
                feature
            )

        return out

    def _candidate_score(
        self,
        frame_index: int,
        box: np.ndarray,
        embedding: Optional[np.ndarray],
        histogram: Optional[np.ndarray],
        record: GlobalIdentityRecord,
    ) -> Dict[str, float]:
        cosine = _cosine_similarity(
            embedding,
            record.embedding,
        )

        if not np.isfinite(
            cosine
        ):
            return {
                "cosine": float("nan"),
                "color": 0.5,
                "size": 0.0,
                "time": 0.0,
                "border_bonus": 0.0,
                "combined": float("-inf"),
                "gap_frames": float(
                    frame_index
                    - record.last_seen_frame
                ),
                "gap_seconds": float(
                    (
                        frame_index
                        - record.last_seen_frame
                    )
                    / self.frame_rate
                ),
            }

        color = _histogram_similarity(
            histogram,
            record.histogram,
        )

        size = _size_similarity(
            box,
            record.last_box,
        )

        gap_frames = max(
            0,
            int(
                frame_index
                - record.last_seen_frame
            ),
        )

        gap_seconds = (
            gap_frames
            / self.frame_rate
        )

        time_similarity = float(
            np.exp(
                -gap_seconds
                / max(
                    GLOBAL_ID_MEMORY_SECONDS,
                    1e-6,
                )
            )
        )

        current_side = _border_side(
            box,
            self.frame_width,
            self.frame_height,
        )

        border_bonus = (
            GLOBAL_REID_BORDER_BONUS
            if (
                current_side != "center"
                and
                current_side
                == record.last_border_side
            )
            else 0.0
        )

        combined = (
            GLOBAL_REID_WEIGHT_APPEARANCE
            * cosine
            +
            GLOBAL_REID_WEIGHT_COLOR
            * color
            +
            GLOBAL_REID_WEIGHT_SIZE
            * size
            +
            GLOBAL_REID_WEIGHT_TIME
            * time_similarity
            +
            border_bonus
        )

        return {
            "cosine": float(
                cosine
            ),
            "color": float(
                color
            ),
            "size": float(
                size
            ),
            "time": float(
                time_similarity
            ),
            "border_bonus": float(
                border_bonus
            ),
            "combined": float(
                combined
            ),
            "gap_frames": float(
                gap_frames
            ),
            "gap_seconds": float(
                gap_seconds
            ),
        }

    def _new_global_identity(
        self,
        local_id: int,
        frame_index: int,
        box: np.ndarray,
        embedding: Optional[np.ndarray],
        histogram: Optional[np.ndarray],
    ) -> int:
        gid = int(
            self.next_global_id
        )

        self.next_global_id += 1

        self.records[gid] = GlobalIdentityRecord(
            global_id=gid,
            embedding=_l2_normalize(
                embedding
            ),
            histogram=(
                None
                if histogram is None
                else np.asarray(
                    histogram,
                    np.float32,
                ).copy()
            ),
            last_box=np.asarray(
                box,
                np.float32,
            ).copy(),
            last_seen_frame=int(
                frame_index
            ),
            last_local_id=int(
                local_id
            ),
            active_local_id=int(
                local_id
            ),
            last_border_side=_border_side(
                box,
                self.frame_width,
                self.frame_height,
            ),
            seen_updates=1,
        )

        self.local_to_global[
            int(local_id)
        ] = gid

        self.unique_global_ids.add(
            gid
        )

        self.new_identity_events += 1

        return gid

    def _update_record(
        self,
        gid: int,
        local_id: int,
        frame_index: int,
        box: np.ndarray,
        embedding: Optional[np.ndarray],
        histogram: Optional[np.ndarray],
    ) -> None:
        record = self.records[
            int(gid)
        ]

        if embedding is not None:
            emb = _l2_normalize(
                embedding
            )

            if emb is not None:
                if record.embedding is None:
                    record.embedding = emb
                else:
                    blended = (
                        GLOBAL_REID_EMA_ALPHA
                        * record.embedding
                        +
                        (
                            1.0
                            - GLOBAL_REID_EMA_ALPHA
                        )
                        * emb
                    )

                    record.embedding = _l2_normalize(
                        blended
                    )

        if histogram is not None:
            hist = np.asarray(
                histogram,
                np.float32,
            )

            if record.histogram is None:
                record.histogram = hist.copy()
            else:
                blended_hist = (
                    GLOBAL_REID_EMA_ALPHA
                    * record.histogram
                    +
                    (
                        1.0
                        - GLOBAL_REID_EMA_ALPHA
                    )
                    * hist
                )

                norm = float(
                    np.linalg.norm(
                        blended_hist
                    )
                )

                if norm > 1e-12:
                    record.histogram = (
                        blended_hist
                        / norm
                    )

        record.last_box = np.asarray(
            box,
            np.float32,
        ).copy()

        record.last_seen_frame = int(
            frame_index
        )

        record.last_local_id = int(
            local_id
        )

        record.active_local_id = int(
            local_id
        )

        record.last_border_side = _border_side(
            box,
            self.frame_width,
            self.frame_height,
        )

        record.seen_updates += 1

    def update(
        self,
        frame_index: int,
        frame: np.ndarray,
        local_tracks: TrackBatch,
    ) -> Tuple[
        TrackBatch,
        Dict[str, Any],
    ]:
        if len(
            local_tracks
        ) == 0:
            for record in self.records.values():
                record.active_local_id = None

            return (
                TrackBatch.empty(),
                {
                    "local_ids": [],
                    "global_ids": [],
                    "recoveries_this_frame": 0,
                },
            )

        boxes = np.asarray(
            local_tracks.boxes,
            np.float32,
        )

        local_ids = np.asarray(
            local_tracks.ids,
            np.int32,
        )

        observed_local_ids = {
            int(x)
            for x in local_ids.tolist()
        }

        self.unique_local_ids.update(
            observed_local_ids
        )

        # Any global identity whose local track is absent from the current
        # tracker output becomes eligible for later recovery.
        for record in self.records.values():
            if (
                record.active_local_id
                is not None
                and
                int(
                    record.active_local_id
                )
                not in observed_local_ids
            ):
                record.active_local_id = None

        global_ids = np.full(
            len(local_tracks),
            -1,
            dtype=np.int32,
        )

        new_indices = []

        # First preserve already-known Local -> Global mappings.
        for i, local_id in enumerate(
            local_ids.tolist()
        ):
            local_id = int(
                local_id
            )

            gid = self.local_to_global.get(
                local_id
            )

            if gid is None:
                new_indices.append(
                    i
                )
                continue

            record = self.records.get(
                int(gid)
            )

            if record is None:
                self.local_to_global.pop(
                    local_id,
                    None,
                )

                new_indices.append(
                    i
                )
                continue

            # Prevent one Global ID from being simultaneously attached to
            # two visible Local IDs.
            if (
                record.active_local_id
                is not None
                and
                int(
                    record.active_local_id
                )
                != local_id
                and
                int(
                    record.active_local_id
                )
                in observed_local_ids
            ):
                self.local_to_global.pop(
                    local_id,
                    None,
                )

                new_indices.append(
                    i
                )
                continue

            global_ids[i] = int(
                gid
            )

            record.active_local_id = int(
                local_id
            )

        # Extract appearance for every new local track and periodically for
        # existing tracks so identity prototypes remain current.
        refresh_indices = set(
            new_indices
        )

        if (
            int(frame_index)
            % max(
                1,
                int(
                    GLOBAL_REID_FEATURE_INTERVAL_FRAMES
                ),
            )
            == 0
        ):
            refresh_indices.update(
                range(
                    len(local_tracks)
                )
            )

        feature_map = self._extract_features(
            frame,
            boxes,
            sorted(
                refresh_indices
            ),
        )

        histogram_map = {
            int(i): _crop_hsv_histogram(
                frame,
                boxes[int(i)],
            )
            for i in refresh_indices
        }

        # Build the candidate gallery only from inactive, non-expired IDs.
        candidate_gids = []

        for gid, record in self.records.items():
            gap_frames = (
                int(frame_index)
                - int(
                    record.last_seen_frame
                )
            )

            if gap_frames < 1:
                continue

            if gap_frames > self.memory_frames:
                continue

            if (
                record.active_local_id
                is not None
                and
                int(
                    record.active_local_id
                )
                in observed_local_ids
            ):
                continue

            candidate_gids.append(
                int(gid)
            )

        # Compute all strong candidate pairs, then assign greedily from the
        # highest score down. This guarantees one-to-one recovery.
        candidate_pairs = []
        row_candidate_scores: Dict[
            int,
            List[float],
        ] = defaultdict(
            list
        )

        for idx in new_indices:
            embedding = feature_map.get(
                int(idx)
            )

            histogram = histogram_map.get(
                int(idx)
            )

            if embedding is None:
                self.no_feature_rejections += 1
                continue

            for gid in candidate_gids:
                record = self.records[
                    int(gid)
                ]

                details = self._candidate_score(
                    int(frame_index),
                    boxes[int(idx)],
                    embedding,
                    histogram,
                    record,
                )

                combined = float(
                    details["combined"]
                )

                row_candidate_scores[
                    int(idx)
                ].append(
                    combined
                )

                if (
                    np.isfinite(
                        details["cosine"]
                    )
                    and
                    details["cosine"]
                    >= GLOBAL_REID_MIN_COSINE
                    and
                    combined
                    >= GLOBAL_REID_MIN_COMBINED_SCORE
                ):
                    candidate_pairs.append(
                        (
                            combined,
                            int(idx),
                            int(gid),
                            details,
                        )
                    )

        candidate_pairs.sort(
            key=lambda item: item[0],
            reverse=True,
        )

        used_indices = set()
        used_gids = set()
        recoveries_this_frame = 0

        for (
            combined,
            idx,
            gid,
            details,
        ) in candidate_pairs:
            if (
                idx in used_indices
                or gid in used_gids
            ):
                continue

            scores = sorted(
                row_candidate_scores.get(
                    idx,
                    [],
                ),
                reverse=True,
            )

            runner_up = (
                scores[1]
                if len(scores) > 1
                else float("-inf")
            )

            margin = (
                float(combined)
                - float(runner_up)
                if np.isfinite(
                    runner_up
                )
                else 1.0
            )

            if (
                margin
                < GLOBAL_REID_MIN_MARGIN
            ):
                self.ambiguous_rejections += 1
                continue

            local_id = int(
                local_ids[idx]
            )

            record = self.records[
                gid
            ]

            previous_local_id = int(
                record.last_local_id
            )

            # Remove stale local aliases pointing to the same Global ID.
            for old_local, old_gid in list(
                self.local_to_global.items()
            ):
                if (
                    int(old_gid) == gid
                    and
                    int(old_local) != local_id
                    and
                    int(old_local)
                    not in observed_local_ids
                ):
                    self.local_to_global.pop(
                        int(old_local),
                        None,
                    )

            self.local_to_global[
                local_id
            ] = gid

            global_ids[
                idx
            ] = gid

            used_indices.add(
                idx
            )

            used_gids.add(
                gid
            )

            recoveries_this_frame += 1
            self.recovered_identity_events += 1

            event = {
                "frame": int(
                    frame_index
                ),
                "event": "global_id_recovered",
                "local_id": local_id,
                "previous_local_id": previous_local_id,
                "global_id": gid,
                "cosine": float(
                    details["cosine"]
                ),
                "combined_score": float(
                    combined
                ),
                "margin": float(
                    margin
                ),
                "gap_frames": int(
                    details["gap_frames"]
                ),
                "gap_seconds": float(
                    details["gap_seconds"]
                ),
                "color_similarity": float(
                    details["color"]
                ),
                "size_similarity": float(
                    details["size"]
                ),
                "time_similarity": float(
                    details["time"]
                ),
                "last_border_side": str(
                    record.last_border_side
                ),
                "current_border_side": _border_side(
                    boxes[idx],
                    self.frame_width,
                    self.frame_height,
                ),
            }

            self.recovery_events.append(
                event
            )

        # Any still-unmapped local track receives a new Global ID.
        for idx in new_indices:
            if global_ids[
                idx
            ] >= 0:
                continue

            local_id = int(
                local_ids[idx]
            )

            gid = self._new_global_identity(
                local_id=local_id,
                frame_index=int(
                    frame_index
                ),
                box=boxes[idx],
                embedding=feature_map.get(
                    int(idx)
                ),
                histogram=histogram_map.get(
                    int(idx)
                ),
            )

            global_ids[
                idx
            ] = gid

        # Update active identity prototypes.
        for idx, (
            local_id,
            gid,
        ) in enumerate(
            zip(
                local_ids.tolist(),
                global_ids.tolist(),
            )
        ):
            self._update_record(
                gid=int(
                    gid
                ),
                local_id=int(
                    local_id
                ),
                frame_index=int(
                    frame_index
                ),
                box=boxes[idx],
                embedding=feature_map.get(
                    int(idx)
                ),
                histogram=histogram_map.get(
                    int(idx)
                ),
            )

            self.unique_global_ids.add(
                int(gid)
            )

        global_tracks = TrackBatch(
            boxes=boxes.copy(),
            ids=global_ids.astype(
                np.int32
            ),
            scores=np.asarray(
                local_tracks.scores,
                np.float32,
            ).copy(),
        )

        return (
            global_tracks,
            {
                "local_ids": local_ids.astype(
                    int
                ).tolist(),
                "global_ids": global_ids.astype(
                    int
                ).tolist(),
                "recoveries_this_frame": int(
                    recoveries_this_frame
                ),
            },
        )

    def summary(
        self,
    ) -> Dict[str, Any]:
        return {
            "global_id_recovery_enabled": bool(
                ENABLE_GLOBAL_ID_RECOVERY
                and self.encoder is not None
            ),
            "global_reid_model": GLOBAL_REID_MODEL,
            "memory_seconds": float(
                GLOBAL_ID_MEMORY_SECONDS
            ),
            "unique_local_ids": int(
                len(
                    self.unique_local_ids
                )
            ),
            "unique_global_ids": int(
                len(
                    self.unique_global_ids
                )
            ),
            "new_global_id_events": int(
                self.new_identity_events
            ),
            "recovered_global_id_events": int(
                self.recovered_identity_events
            ),
            "ambiguous_reid_rejections": int(
                self.ambiguous_rejections
            ),
            "no_feature_rejections": int(
                self.no_feature_rejections
            ),
            "local_to_global_reduction": int(
                len(
                    self.unique_local_ids
                )
                -
                len(
                    self.unique_global_ids
                )
            ),
        }


print(
    "Long-Term Global-ID layer: READY"
)
print(
    "ReID model:",
    GLOBAL_REID_MODEL,
)
print(
    "Identity memory:",
    f"{GLOBAL_ID_MEMORY_SECONDS:.0f} seconds",
)

Long-Term Global-ID layer: READY
ReID model: yolo26n-reid.onnx
Identity memory: 60 seconds


## 12. Detection/MOT metrics, video, and segment utilities

In [37]:
def xyxy_to_xywh(boxes: np.ndarray) -> np.ndarray:
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    out = boxes.copy()
    if len(out):
        out[:, 2] = boxes[:, 2] - boxes[:, 0]; out[:, 3] = boxes[:, 3] - boxes[:, 1]
    return out


def box_iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a, b = np.asarray(a, np.float32).reshape(-1, 4), np.asarray(b, np.float32).reshape(-1, 4)
    if not len(a) or not len(b): return np.zeros((len(a), len(b)), np.float32)
    lt, rb = np.maximum(a[:, None, :2], b[None, :, :2]), np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.clip(area_a[:, None] + area_b[None, :] - inter, 1e-9, None)


def match_boxes(gt_boxes: np.ndarray, pred_boxes: np.ndarray, iou_threshold: float):
    ious = box_iou_matrix(gt_boxes, pred_boxes)
    if not ious.size: return [], set(), set()
    gi, pi = linear_sum_assignment(-ious); matches, mg, mp = [], set(), set()
    for g, p in zip(gi, pi):
        iou = float(ious[g, p])
        if iou >= iou_threshold:
            matches.append((int(g), int(p), iou)); mg.add(int(g)); mp.add(int(p))
    return matches, mg, mp


def detection_operating_metrics(gt_df: pd.DataFrame, detections_by_original_frame: Dict[int, DetectionBatch]) -> Dict[str, Any]:
    tp = fp = fn = occ_total = occ_hit = vis_total = vis_hit = 0
    bins = {k: {"total": 0, "hit": 0} for k in ["h_lt16", "h_16_31", "h_32_95", "h_ge96"]}
    for original_frame, frame_gt in gt_df.groupby("frame"):
        gt_boxes = frame_gt[["xmin", "ymin", "xmax", "ymax"]].to_numpy(np.float32)
        det = detections_by_original_frame.get(int(original_frame), DetectionBatch.empty())
        matches, mg, mp = match_boxes(gt_boxes, det.boxes, DETECTION_EVAL_IOU)
        tp += len(matches); fp += len(det.boxes) - len(mp); fn += len(gt_boxes) - len(mg)
        reset = frame_gt.reset_index(drop=True)
        for idx, row in reset.iterrows():
            hit = int(idx) in mg
            if int(row["occluded"]) == 1:
                occ_total += 1; occ_hit += int(hit)
            else:
                vis_total += 1; vis_hit += int(hit)
            h = float(row["ymax"] - row["ymin"])
            key = "h_lt16" if h < 16 else "h_16_31" if h < 32 else "h_32_95" if h < 96 else "h_ge96"
            bins[key]["total"] += 1; bins[key]["hit"] += int(hit)
    precision = tp / max(tp + fp, 1); recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    out = {
        "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1,
        "visible_recall": vis_hit / max(vis_total, 1), "occluded_recall": occ_hit / max(occ_total, 1),
        "visible_gt": vis_total, "occluded_gt": occ_total,
    }
    for key, val in bins.items():
        out[f"{key}_gt"] = val["total"]; out[f"{key}_recall"] = val["hit"] / max(val["total"], 1)
    return out


def load_segment_gt(segment: pd.Series) -> pd.DataFrame:
    labels = parse_okutama_label_file(Path(segment["label_path"]))
    return labels[(labels["frame"] >= int(segment["start_frame"])) & (labels["frame"] <= int(segment["end_frame"]))].copy()


def frame_gt_dict(gt_df: pd.DataFrame):
    return {int(f): g.copy() for f, g in gt_df.groupby("frame")}


def open_video_at(video_path: Path, zero_based_index: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): raise RuntimeError(f"Could not open video: {video_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(zero_based_index))
    return cap


def read_exact_segment_frames(segment: pd.Series):
    cap = open_video_at(Path(segment["video_path"]), int(segment["video_start_index"]))
    try:
        for local_idx in range(int(segment["length"])):
            ok, frame = cap.read()
            if not ok or frame is None:
                raise RuntimeError(f"Video ended early at local frame {local_idx + 1}: {segment['video_path']}")
            yield local_idx + 1, int(segment["start_frame"]) + local_idx, frame
    finally:
        cap.release()


def write_mot_gt(seq_name: str, segment: pd.Series, gt_df: pd.DataFrame, gt_benchmark_root: Path):
    seq_dir = gt_benchmark_root / seq_name; gt_dir = seq_dir / "gt"; gt_dir.mkdir(parents=True, exist_ok=True)
    start = int(segment["start_frame"]); lines = []
    # MOTChallenge expects positive target IDs. Identity labels are arbitrary, so
    # remapping Okutama IDs to contiguous 1-based IDs preserves the tracking task.
    original_ids = sorted(int(x) for x in gt_df["track_id"].unique().tolist())
    gt_id_map = {orig: idx + 1 for idx, orig in enumerate(original_ids)}
    write_json(seq_dir / "okutama_to_mot_gt_id_map.json", gt_id_map)
    for row in gt_df.itertuples():
        local_frame = int(row.frame) - start + 1
        x, y = float(row.xmin), float(row.ymin); w, h = float(row.xmax-row.xmin), float(row.ymax-row.ymin)
        visibility = 0.5 if int(row.occluded) == 1 else 1.0
        mot_id = gt_id_map[int(row.track_id)]
        lines.append(f"{local_frame},{mot_id},{x:.3f},{y:.3f},{w:.3f},{h:.3f},1,1,{visibility:.3f},-1")
    (gt_dir / "gt.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    seqinfo = f'''[Sequence]
name={seq_name}
imDir=img1
frameRate={float(segment['fps']):.6f}
seqLength={int(segment['length'])}
imWidth={int(segment['width'])}
imHeight={int(segment['height'])}
imExt=.jpg
'''
    (seq_dir / "seqinfo.ini").write_text(seqinfo, encoding="utf-8")


def append_mot_prediction_line(lines: List[str], local_frame: int, track_id: int, box: Sequence[float], score: float):
    x1, y1, x2, y2 = map(float, box); w, h = x2 - x1, y2 - y1
    lines.append(f"{local_frame},{track_id},{x1:.3f},{y1:.3f},{w:.3f},{h:.3f},{score:.6f},-1,-1,-1")


def id_color(track_id: int):
    rng = np.random.default_rng(int(track_id) + 12345)
    return tuple(int(x) for x in rng.integers(60, 240, size=3).tolist())


def draw_tracking_frame(frame, tracks: TrackBatch, gt_frame: Optional[pd.DataFrame], trails, model_name, tracker_name,
                        local_frame, detector_ms, tracker_ms):
    canvas = frame.copy()
    if SHOW_GROUND_TRUTH_ON_VIDEO and gt_frame is not None:
        for row in gt_frame.itertuples():
            x1, y1, x2, y2 = map(int, [row.xmin, row.ymin, row.xmax, row.ymax])
            cv2.rectangle(canvas, (x1, y1), (x2, y2), (235, 235, 235), 1)
            cv2.putText(canvas, f"GT {int(row.track_id)}", (x1, max(15, y1-4)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.45, (235,235,235), 1, cv2.LINE_AA)
    for box, tid, score in zip(tracks.boxes, tracks.ids, tracks.scores):
        x1, y1, x2, y2 = map(int, box); color = id_color(int(tid))
        cv2.rectangle(canvas, (x1,y1), (x2,y2), color, 2)
        cv2.putText(canvas, f"ID {int(tid)} {float(score):.2f}", (x1, max(18, y1-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
        if SHOW_TRACK_TRAILS:
            trails[int(tid)].append((int((x1+x2)/2), int((y1+y2)/2)))
            pts = list(trails[int(tid)])
            for p1, p2 in zip(pts[:-1], pts[1:]): cv2.line(canvas, p1, p2, color, 2, cv2.LINE_AA)
    core_ms = detector_ms + tracker_ms; core_fps = 1000/core_ms if core_ms > 0 else 0
    info = [f"{model_name} + {tracker_name}", f"Frame: {local_frame}", f"Tracks: {len(tracks)}",
            f"Detector: {detector_ms:.2f} ms", f"Tracker: {tracker_ms:.2f} ms",
            f"Core: {core_ms:.2f} ms ({core_fps:.1f} FPS)"]
    y = 28
    for line in info:
        cv2.putText(canvas, line, (15,y), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255,255,255), 2, cv2.LINE_AA); y += 24
    return canvas


print("Metrics/MOT/video utilities: READY")


Metrics/MOT/video utilities: READY


## 13. Prepare GT and execute the 3-detector × 2-tracker experiment matrix
Each detector runs once per clip. Its saved detections are replayed into both trackers,
guaranteeing that ByteTrack and BoT-SORT receive identical detector outputs.

In [38]:
TRACKING_BENCHMARK_NAME = "OKUTAMA_STEP5"
TRACKING_SPLIT = "train"
TRACKEVAL_GT_ROOT = TRACKEVAL_DIR / "mot_gt"
TRACKEVAL_TRACKERS_ROOT = TRACKEVAL_DIR / "mot_trackers"
TRACKEVAL_OUTPUT_ROOT = TRACKEVAL_DIR / "results"
GT_BENCHMARK_ROOT = TRACKEVAL_GT_ROOT / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}"
GT_BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
(TRACKEVAL_GT_ROOT / "seqmaps").mkdir(parents=True, exist_ok=True)
TRACKEVAL_TRACKERS_ROOT.mkdir(parents=True, exist_ok=True); TRACKEVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

segment_gt_cache = {}
for _, segment in SELECTED_SEGMENTS.iterrows():
    seq = str(segment["sequence_name"]); gt_df = load_segment_gt(segment); segment_gt_cache[seq] = gt_df
    write_mot_gt(seq, segment, gt_df, GT_BENCHMARK_ROOT)
seqmap_path = TRACKEVAL_GT_ROOT / "seqmaps" / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}.txt"
seqmap_path.write_text("name\n" + "\n".join(SELECTED_SEGMENTS["sequence_name"].astype(str)) + "\n", encoding="utf-8")


def detection_cache_paths(model_name: str, seq: str):
    root = RUNS_DIR / safe_slug(model_name) / "detector_cache" / seq; root.mkdir(parents=True, exist_ok=True)
    return {"root": root, "jsonl": root/"detections.jsonl", "summary": root/"detection_summary.json",
            "timings": root/"detector_timings.csv", "complete": root/"_COMPLETE.json"}


def serialize_detection(local_frame, original_frame, b: DetectionBatch):
    return {"local_frame": int(local_frame), "original_frame": int(original_frame),
            "boxes": b.boxes.astype(float).tolist(), "scores": b.scores.astype(float).tolist(),
            "preprocess_ms": float(b.preprocess_ms), "inference_ms": float(b.inference_ms),
            "postprocess_ms": float(b.postprocess_ms), "total_ms": float(b.total_ms)}


def deserialize_detection(row):
    boxes = np.asarray(row.get("boxes", []), np.float32).reshape(-1,4); scores = np.asarray(row.get("scores", []), np.float32)
    return DetectionBatch(boxes, scores, np.zeros((len(boxes),), np.float32), float(row.get("preprocess_ms",0)),
                          float(row.get("inference_ms",0)), float(row.get("postprocess_ms",0)), float(row.get("total_ms",0)))


def load_detection_cache(path: Path):
    rows = [json.loads(x) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]
    return rows, {int(r["original_frame"]): deserialize_detection(r) for r in rows}


def run_detector_for_segment(detector, model_name: str, segment: pd.Series):
    seq = str(segment["sequence_name"]); paths = detection_cache_paths(model_name, seq)
    if RESUME_COMPLETED_DETECTION_CACHE and paths["complete"].exists() and paths["jsonl"].exists() and paths["summary"].exists():
        print("Detection cache complete:", model_name, seq)
        rows, by_original = load_detection_cache(paths["jsonl"])
        return rows, by_original, read_json(paths["summary"])

    cap = open_video_at(Path(segment["video_path"]), int(segment["video_start_index"])); ok, first_frame = cap.read(); cap.release()
    if not ok or first_frame is None: raise RuntimeError(f"Could not read warmup frame: {seq}")
    for _ in range(WARMUP_RUNS): _ = detector.predict(first_frame)
    reset_cuda_peak(); rows, by_original, timing_rows = [], {}, []
    for local_frame, original_frame, frame in tqdm(read_exact_segment_frames(segment), total=int(segment["length"]),
                                                   desc=f"Detect {model_name} | {seq}", leave=False):
        b = detector.predict(frame); by_original[original_frame] = b
        row = serialize_detection(local_frame, original_frame, b); rows.append(row)
        timing_rows.append({"local_frame": local_frame, "original_frame": original_frame, "num_detections": len(b),
                            "preprocess_ms": b.preprocess_ms, "inference_ms": b.inference_ms,
                            "postprocess_ms": b.postprocess_ms, "total_ms": b.total_ms})
    det_metrics = detection_operating_metrics(segment_gt_cache[seq], by_original)
    total_times = [r["total_ms"] for r in timing_rows]; inf_times = [r["inference_ms"] for r in timing_rows]
    mean_total = nanmean(total_times)
    summary = {"model": model_name, "sequence_name": seq, "frames": len(rows),
               "mean_detector_total_ms": mean_total, "p50_detector_total_ms": percentile(total_times,50),
               "p95_detector_total_ms": percentile(total_times,95), "mean_detector_inference_ms": nanmean(inf_times),
               "detector_fps": 1000/mean_total if mean_total > 0 else float("nan"), "peak_cuda_mib": peak_cuda_mib(),
               **det_metrics}
    if SAVE_DETECTION_CACHE_JSONL:
        with paths["jsonl"].open("w", encoding="utf-8") as f:
            for row in rows: f.write(json.dumps(row) + "\n")
    pd.DataFrame(timing_rows).to_csv(paths["timings"], index=False); write_json(paths["summary"], summary)
    write_json(paths["complete"], {"timestamp_utc": utc_now_iso(), "frames": len(rows)})
    return rows, by_original, summary


def tracker_run_paths(model_name, tracker_name, seq):
    system = safe_slug(f"{model_name}__{tracker_name}"); root = RUNS_DIR / system / seq; root.mkdir(parents=True, exist_ok=True)
    return system, {
        "root": root,
        "video": root / "annotated_tracking.mp4",
        "frames_csv": root / "frame_metrics.csv",
        "tracks_jsonl": root / "tracks.jsonl",
        "summary": root / "run_summary.json",
        "mot_prediction": root / "mot_predictions_global_id.txt",
        "local_mot_prediction": root / "mot_predictions_local_id_raw.txt",
        "reid_events_csv": root / "global_id_recovery_events.csv",
        "complete": root / "_COMPLETE.json",
    }


def run_tracker_for_segment(
    model_name,
    tracker_name,
    segment,
    detections_by_original,
    detector_summary,
):
    seq = str(
        segment["sequence_name"]
    )

    system, paths = tracker_run_paths(
        model_name,
        tracker_name,
        seq,
    )

    global_dir = (
        TRACKEVAL_TRACKERS_ROOT
        / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}"
        / system
        / "data"
    )

    global_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    global_mot = (
        global_dir
        / f"{seq}.txt"
    )

    if (
        RESUME_COMPLETED_TRACKER_RUNS
        and paths["complete"].exists()
        and paths["summary"].exists()
        and global_mot.exists()
    ):
        print(
            "Tracker run complete:",
            system,
            seq,
        )

        return read_json(
            paths["summary"]
        )

    fps = float(
        segment["fps"]
    )
    width = int(
        segment["width"]
    )
    height = int(
        segment["height"]
    )

    tracker = CommonTracker(
        model_name,
        tracker_name,
        fps,
    )

    tracker_config_audit = (
        tracker.audit_dict()
    )

    identity_manager = (
        GlobalIdentityManager(
            frame_rate=fps,
            frame_width=width,
            frame_height=height,
        )
        if ENABLE_GLOBAL_ID_RECOVERY
        else None
    )

    print(
        f"{model_name} + {tracker_name} | "
        f"FPS={fps:.3f} | "
        f"buffer={tracker_config_audit['actual_tracker_buffer_frames']} frames "
        f"({tracker_config_audit['actual_tracker_buffer_seconds']:.2f} s) | "
        f"global_memory={GLOBAL_ID_MEMORY_SECONDS:.0f} s | "
        f"match={tracker_config_audit['match_thresh']:.2f} | "
        f"high={tracker_config_audit['track_high_thresh']:.2f} | "
        f"new={tracker_config_audit['new_track_thresh']:.2f}"
    )

    gt_by = frame_gt_dict(
        segment_gt_cache[
            seq
        ]
    )

    writer = None

    if SAVE_ANNOTATED_VIDEO:
        writer = cv2.VideoWriter(
            str(
                paths["video"]
            ),
            cv2.VideoWriter_fourcc(
                *"mp4v"
            ),
            fps,
            (
                width,
                height,
            ),
        )

        if not writer.isOpened():
            raise RuntimeError(
                f"Could not create video: {paths['video']}"
            )

    trails = defaultdict(
        lambda: deque(
            maxlen=TRACK_TRAIL_LENGTH
        )
    )

    frame_rows = []
    track_rows = []
    mot_lines = []
    local_mot_lines = []

    tracker_times = []
    identity_times = []
    core_times = []

    wall_start = time.perf_counter()

    try:
        for (
            local_frame,
            original_frame,
            frame,
        ) in tqdm(
            read_exact_segment_frames(
                segment
            ),
            total=int(
                segment["length"]
            ),
            desc=(
                f"Track {system} | {seq}"
            ),
            leave=False,
        ):
            det = detections_by_original[
                original_frame
            ]

            track_start = time.perf_counter()

            local_tracks = tracker.update(
                det,
                frame,
            )

            tracker_ms = (
                time.perf_counter()
                - track_start
            ) * 1000.0

            identity_start = time.perf_counter()

            if identity_manager is not None:
                (
                    tracks,
                    identity_meta,
                ) = identity_manager.update(
                    frame_index=int(
                        local_frame
                    ),
                    frame=frame,
                    local_tracks=local_tracks,
                )
            else:
                tracks = local_tracks

                identity_meta = {
                    "local_ids": local_tracks.ids.astype(
                        int
                    ).tolist(),
                    "global_ids": local_tracks.ids.astype(
                        int
                    ).tolist(),
                    "recoveries_this_frame": 0,
                }

            cuda_sync()

            identity_ms = (
                time.perf_counter()
                - identity_start
            ) * 1000.0

            core_ms = (
                float(
                    det.total_ms
                )
                + tracker_ms
                + identity_ms
            )

            tracker_times.append(
                tracker_ms
            )

            identity_times.append(
                identity_ms
            )

            core_times.append(
                core_ms
            )

            # Raw tracker output for audit.
            for (
                box,
                local_tid,
                score,
            ) in zip(
                local_tracks.boxes,
                local_tracks.ids,
                local_tracks.scores,
            ):
                append_mot_prediction_line(
                    local_mot_lines,
                    local_frame,
                    int(
                        local_tid
                    ),
                    box,
                    float(
                        score
                    ),
                )

            # Final formal output uses Global IDs.
            for (
                box,
                global_tid,
                score,
            ) in zip(
                tracks.boxes,
                tracks.ids,
                tracks.scores,
            ):
                append_mot_prediction_line(
                    mot_lines,
                    local_frame,
                    int(
                        global_tid
                    ),
                    box,
                    float(
                        score
                    ),
                )

            frame_rows.append(
                {
                    "local_frame": local_frame,
                    "original_frame": original_frame,
                    "detections": len(
                        det
                    ),
                    "local_tracks": len(
                        local_tracks
                    ),
                    "global_tracks": len(
                        tracks
                    ),
                    "global_recoveries": int(
                        identity_meta[
                            "recoveries_this_frame"
                        ]
                    ),
                    "detector_ms": det.total_ms,
                    "tracker_ms": tracker_ms,
                    "global_identity_ms": identity_ms,
                    "core_pipeline_ms": core_ms,
                    "core_pipeline_fps": (
                        1000.0
                        / core_ms
                        if core_ms > 0
                        else np.nan
                    ),
                }
            )

            track_rows.append(
                {
                    "local_frame": local_frame,
                    "original_frame": original_frame,
                    "local_track_ids": [
                        int(x)
                        for x in identity_meta[
                            "local_ids"
                        ]
                    ],
                    "global_track_ids": [
                        int(x)
                        for x in identity_meta[
                            "global_ids"
                        ]
                    ],
                    "boxes": tracks.boxes.astype(
                        float
                    ).tolist(),
                    "scores": tracks.scores.astype(
                        float
                    ).tolist(),
                }
            )

            if writer is not None:
                writer.write(
                    draw_tracking_frame(
                        frame,
                        tracks,
                        gt_by.get(
                            original_frame
                        ),
                        trails,
                        model_name,
                        tracker_name,
                        local_frame,
                        float(
                            det.total_ms
                        ),
                        tracker_ms
                        + identity_ms,
                    )
                )

    finally:
        if writer is not None:
            writer.release()

    wall = (
        time.perf_counter()
        - wall_start
    )

    mot_text = (
        "\n".join(
            mot_lines
        )
        + (
            "\n"
            if mot_lines
            else ""
        )
    )

    local_mot_text = (
        "\n".join(
            local_mot_lines
        )
        + (
            "\n"
            if local_mot_lines
            else ""
        )
    )

    paths[
        "mot_prediction"
    ].write_text(
        mot_text,
        encoding="utf-8",
    )

    paths[
        "local_mot_prediction"
    ].write_text(
        local_mot_text,
        encoding="utf-8",
    )

    # TrackEval receives the Global-ID version.
    global_mot.write_text(
        mot_text,
        encoding="utf-8",
    )

    if SAVE_FRAME_CSV:
        pd.DataFrame(
            frame_rows
        ).to_csv(
            paths[
                "frames_csv"
            ],
            index=False,
        )

    if SAVE_TRACK_JSONL:
        with paths[
            "tracks_jsonl"
        ].open(
            "w",
            encoding="utf-8",
        ) as f:
            for row in track_rows:
                f.write(
                    json.dumps(
                        row
                    )
                    + "\n"
                )

    if identity_manager is not None:
        recovery_df = pd.DataFrame(
            identity_manager.recovery_events
        )

        recovery_df.to_csv(
            paths[
                "reid_events_csv"
            ],
            index=False,
        )

        identity_summary = (
            identity_manager.summary()
        )

    else:
        pd.DataFrame().to_csv(
            paths[
                "reid_events_csv"
            ],
            index=False,
        )

        identity_summary = {
            "global_id_recovery_enabled": False,
            "unique_local_ids": int(
                len(
                    {
                        int(x)
                        for row in track_rows
                        for x in row[
                            "local_track_ids"
                        ]
                    }
                )
            ),
            "unique_global_ids": int(
                len(
                    {
                        int(x)
                        for row in track_rows
                        for x in row[
                            "global_track_ids"
                        ]
                    }
                )
            ),
            "recovered_global_id_events": 0,
            "ambiguous_reid_rejections": 0,
            "no_feature_rejections": 0,
            "local_to_global_reduction": 0,
        }

    mean_tracker = nanmean(
        tracker_times
    )

    mean_identity = nanmean(
        identity_times
    )

    mean_core = nanmean(
        core_times
    )

    summary = {
        "system_name": system,
        "model": model_name,
        "tracker": tracker_name,
        "sequence_name": seq,
        "frames": len(
            frame_rows
        ),
        "mean_tracker_ms": mean_tracker,
        "p50_tracker_ms": percentile(
            tracker_times,
            50,
        ),
        "p95_tracker_ms": percentile(
            tracker_times,
            95,
        ),
        "mean_global_identity_ms": mean_identity,
        "p95_global_identity_ms": percentile(
            identity_times,
            95,
        ),
        "mean_core_pipeline_ms": mean_core,
        "p50_core_pipeline_ms": percentile(
            core_times,
            50,
        ),
        "p95_core_pipeline_ms": percentile(
            core_times,
            95,
        ),
        "core_pipeline_fps": (
            1000.0
            / mean_core
            if mean_core > 0
            else np.nan
        ),
        "tracker_overhead_fraction": (
            mean_tracker
            / mean_core
            if mean_core > 0
            else np.nan
        ),
        "annotated_export_wall_fps": (
            len(
                frame_rows
            )
            / wall
            if wall > 0
            else np.nan
        ),
        "detector_peak_cuda_mib": float(
            detector_summary.get(
                "peak_cuda_mib",
                np.nan,
            )
        ),
        "tracker_config": tracker_config_audit,
        "track_buffer_frames": int(
            tracker_config_audit[
                "actual_tracker_buffer_frames"
            ]
        ),
        "track_buffer_seconds": float(
            tracker_config_audit[
                "actual_tracker_buffer_seconds"
            ]
        ),
        **identity_summary,
        "mot_prediction_path": str(
            global_mot
        ),
        "local_mot_prediction_path": str(
            paths[
                "local_mot_prediction"
            ]
        ),
        "global_id_recovery_events_path": str(
            paths[
                "reid_events_csv"
            ]
        ),
        "annotated_video_path": (
            str(
                paths["video"]
            )
            if SAVE_ANNOTATED_VIDEO
            else None
        ),
    }

    write_json(
        paths[
            "summary"
        ],
        summary,
    )

    write_json(
        paths[
            "complete"
        ],
        {
            "timestamp_utc": utc_now_iso(),
            "frames": len(
                frame_rows
            ),
            "uses_global_ids": bool(
                ENABLE_GLOBAL_ID_RECOVERY
            ),
        },
    )

    return summary

MODELS_TO_RUN = ["YOLO26s", "RT-DETR-R18", "BPD-YOLOn/L-FPN"]
SYSTEM_ERRORS, detector_summary_rows, tracker_summary_rows = {}, [], []
for model_name in MODELS_TO_RUN:
    print("\n" + "#"*100); print("BUILDING DETECTOR:", model_name); print("#"*100)
    detector = None
    try:
        detector = build_detector(model_name)
        for _, segment in SELECTED_SEGMENTS.iterrows():
            _, det_by_original, det_summary = run_detector_for_segment(detector, model_name, segment)
            detector_summary_rows.append(det_summary)
            for tracker_name in TRACKERS_TO_RUN:
                try:
                    tracker_summary_rows.append(run_tracker_for_segment(model_name, tracker_name, segment, det_by_original, det_summary))
                except Exception as exc:
                    key = f"{model_name}__{tracker_name}__{segment['sequence_name']}"; SYSTEM_ERRORS[key] = traceback.format_exc()
                    print("FAILED:", key, "\n", exc)
                    if not CONTINUE_ON_SYSTEM_ERROR: raise
    except Exception as exc:
        key = f"{model_name}__detector"; SYSTEM_ERRORS[key] = traceback.format_exc(); print("DETECTOR FAILED:", model_name, exc)
        if not CONTINUE_ON_SYSTEM_ERROR: raise
    finally:
        if detector is not None: del detector
        gc.collect(); torch.cuda.empty_cache()

DETECTOR_SUMMARIES = pd.DataFrame(detector_summary_rows); TRACKER_RUN_SUMMARIES = pd.DataFrame(tracker_summary_rows)
DETECTOR_SUMMARIES.to_csv(METRICS_DIR/"detector_segment_metrics.csv", index=False)
TRACKER_RUN_SUMMARIES.to_csv(METRICS_DIR/"tracker_runtime_segment_metrics.csv", index=False)
write_json(AUDIT_DIR/"system_errors.json", SYSTEM_ERRORS)
print("Detector/tracker stage finished. Errors:", len(SYSTEM_ERRORS))



####################################################################################################
BUILDING DETECTOR: YOLO26s
####################################################################################################
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading /content/step5_model_cache_v5/YOLO26s/4ab51e0cc1e09828/yolo26s_step5_model.engine for TensorRT inference...


Detect YOLO26s | okutama_2.2.1_f000000_to_000299:   0%|          | 0/300 [00:00<?, ?it/s]

Loading Long-Term ReID encoder: yolo26n-reid.onnx
Loading yolo26n-reid.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CUDAExecutionProvider
YOLO26s + ByteTrack | FPS=29.970 | buffer=90 frames (3.00 s) | global_memory=60 s | match=0.85 | high=0.28 | new=0.32


Track YOLO26s_ByteTrack | okutama_2.2.1_f000000_to_000299:   0%|          | 0/300 [00:00<?, ?it/s]

Loading yolo26n-reid.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CUDAExecutionProvider
YOLO26s + BoT-SORT | FPS=29.970 | buffer=90 frames (3.00 s) | global_memory=60 s | match=0.85 | high=0.28 | new=0.32


...

BPD-YOLOn/L-FPN + ByteTrack | FPS=29.970 | buffer=90 frames (3.00 s) | global_memory=60 s | match=0.85 | high=0.28 | new=0.32


Track BPD-YOLOn_L-FPN_ByteTrack | okutama_1.2.3_f000300_to_000599:   0%|          | 0/300 [00:00<?, ?it/s]

Loading yolo26n-reid.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CUDAExecutionProvider
BPD-YOLOn/L-FPN + BoT-SORT | FPS=29.970 | buffer=90 frames (3.00 s) | global_memory=60 s | match=0.85 | high=0.28 | new=0.32


Track BPD-YOLOn_L-FPN_BoT-SORT | okutama_1.2.3_f000300_to_000599:   0%|          | 0/300 [00:00<?, ?it/s]

Detector/tracker stage finished. Errors: 0


## Official TrackEval — NumPy 2.x compatibility hotfix

TrackEval is now executed through its **direct Python API** rather than the
`run_mot_challenge.py` command-line wrapper.

This preserves the official TrackEval evaluator and metric implementations,
while fixing compatibility with modern NumPy and avoiding the CLI
`OUTPUT_FOLDER` list conversion issue.

The upstream TrackEval source repository is **not modified**.


In [39]:
TRACKEVAL_REPO_ROOT = TOOL_CACHE / "TrackEval"

TRACK_EVAL_EXECUTED = False
TRACK_EVAL_SKIP_REASON = None
TRACK_EVAL_RESULTS = None


def prepare_trackeval_repository():
    """
    Clone TrackEval if needed and record the exact upstream commit.
    """
    if not TRACKEVAL_REPO_ROOT.exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                TRACKEVAL_REPO_URL,
                str(TRACKEVAL_REPO_ROOT),
            ]
        )

    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(TRACKEVAL_REPO_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()

    write_json(
        AUDIT_DIR / "trackeval_source.json",
        {
            "repo": TRACKEVAL_REPO_URL,
            "resolved_commit": commit,
            "execution_mode": "direct_python_api",
            "numpy_compatibility_shim": True,
        },
    )

    return commit


def install_trackeval_numpy_compatibility_shim():
    """
    TrackEval upstream still contains deprecated NumPy aliases such as
    np.float and np.int.

    Modern NumPy removed those aliases.  They originally mapped to Python
    builtins, so this local-process compatibility shim restores only the
    missing names without downgrading NumPy or editing the TrackEval source.

    np.float64 / np.int64 and all normal NumPy scalar types are untouched.
    """
    compatibility_aliases = {
        "float": float,
        "int": int,
        "bool": bool,
        "object": object,
        "str": str,
        "complex": complex,
    }

    installed = {}

    for name, builtin_type in compatibility_aliases.items():
        if name not in np.__dict__:
            setattr(np, name, builtin_type)
            installed[name] = builtin_type.__name__

    write_json(
        AUDIT_DIR / "trackeval_numpy_compatibility.json",
        {
            "numpy_version": np.__version__,
            "installed_aliases": installed,
            "scope": (
                "Current notebook Python process only; "
                "upstream TrackEval source files are not modified."
            ),
        },
    )

    if installed:
        print(
            "TrackEval NumPy compatibility aliases installed:",
            installed,
        )
    else:
        print(
            "TrackEval NumPy compatibility shim: "
            "no missing aliases required."
        )

    return installed


def expected_system_names():
    return [
        safe_slug(f"{model_name}__{tracker_name}")
        for model_name in MODELS_TO_RUN
        for tracker_name in TRACKERS_TO_RUN
    ]


def trackeval_missing_predictions() -> List[str]:
    missing = []

    for system in expected_system_names():
        for seq in SELECTED_SEGMENTS["sequence_name"].astype(str):
            p = (
                TRACKEVAL_TRACKERS_ROOT
                / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}"
                / system
                / "data"
                / f"{seq}.txt"
            )

            if not p.exists():
                missing.append(str(p))

    return missing


def import_trackeval_direct():
    """
    Import the cloned TrackEval repository into this notebook process.
    """
    repo_text = str(TRACKEVAL_REPO_ROOT)

    if repo_text not in sys.path:
        sys.path.insert(0, repo_text)

    importlib.invalidate_caches()

    import trackeval

    return trackeval


def run_trackeval():
    """
    Run TrackEval through its Python API instead of scripts/run_mot_challenge.py.

    Advantages:
      1. NumPy compatibility shim is active in the same process.
      2. OUTPUT_FOLDER remains a normal string instead of being converted
         to a one-element list by the CLI argparse path.
      3. The official TrackEval Evaluator, MotChallenge2DBox dataset class,
         HOTA, CLEAR and Identity implementations are still used.
    """
    global TRACK_EVAL_EXECUTED
    global TRACK_EVAL_SKIP_REASON
    global TRACK_EVAL_RESULTS

    missing = trackeval_missing_predictions()

    if SYSTEM_ERRORS or missing:
        TRACK_EVAL_SKIP_REASON = {
            "system_errors": SYSTEM_ERRORS,
            "missing_prediction_count": len(missing),
            "missing_prediction_sample": missing[:10],
        }

        write_json(
            TRACKEVAL_DIR / "TRACK_EVAL_SKIPPED.json",
            TRACK_EVAL_SKIP_REASON,
        )

        print("=" * 96)
        print("TRACKEVAL SKIPPED")
        print("=" * 96)
        print(
            "The detector/tracker matrix is incomplete, "
            "so TrackEval was not started."
        )
        print("System errors            :", len(SYSTEM_ERRORS))
        print("Missing prediction files :", len(missing))
        print("=" * 96)

        return False

    commit = prepare_trackeval_repository()

    install_trackeval_numpy_compatibility_shim()

    trackeval = import_trackeval_direct()

    systems = expected_system_names()

    # --------------------------------------------------------
    # Evaluator configuration
    # --------------------------------------------------------

    eval_config = trackeval.Evaluator.get_default_eval_config()

    eval_config.update(
        {
            "USE_PARALLEL": False,
            "NUM_PARALLEL_CORES": 1,
            "BREAK_ON_ERROR": True,
            "RETURN_ON_ERROR": False,
            "PRINT_RESULTS": True,
            "PRINT_ONLY_COMBINED": False,
            "PRINT_CONFIG": True,
            "TIME_PROGRESS": True,
            "DISPLAY_LESS_PROGRESS": False,
            "OUTPUT_SUMMARY": True,
            "OUTPUT_EMPTY_CLASSES": True,
            "OUTPUT_DETAILED": True,
            "PLOT_CURVES": True,
        }
    )

    # --------------------------------------------------------
    # MOTChallenge-compatible custom Okutama configuration
    # --------------------------------------------------------

    dataset_config = (
        trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    )

    dataset_config.update(
        {
            "GT_FOLDER": str(TRACKEVAL_GT_ROOT),
            "TRACKERS_FOLDER": str(TRACKEVAL_TRACKERS_ROOT),

            # IMPORTANT:
            # Direct Python API keeps this as a string.
            "OUTPUT_FOLDER": str(TRACKEVAL_OUTPUT_ROOT),

            "TRACKERS_TO_EVAL": systems,
            "CLASSES_TO_EVAL": ["pedestrian"],
            "BENCHMARK": TRACKING_BENCHMARK_NAME,
            "SPLIT_TO_EVAL": TRACKING_SPLIT,
            "INPUT_AS_ZIP": False,
            "PRINT_CONFIG": True,
            "DO_PREPROC": False,
            "TRACKER_SUB_FOLDER": "data",
            "OUTPUT_SUB_FOLDER": "",

            # Use the exact seqmap generated by this notebook.
            "SEQMAP_FOLDER": str(
                TRACKEVAL_GT_ROOT / "seqmaps"
            ),
            "SEQMAP_FILE": str(seqmap_path),

            "SEQ_INFO": None,
            "GT_LOC_FORMAT": "{gt_folder}/{seq}/gt/gt.txt",
            "SKIP_SPLIT_FOL": False,
        }
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics_config = {
        "METRICS": [
            "HOTA",
            "CLEAR",
            "Identity",
        ],
        "THRESHOLD": 0.5,
    }

    evaluator = trackeval.Evaluator(
        eval_config
    )

    dataset_list = [
        trackeval.datasets.MotChallenge2DBox(
            dataset_config
        )
    ]

    metrics_list = [
        trackeval.metrics.HOTA(
            metrics_config
        ),
        trackeval.metrics.CLEAR(
            metrics_config
        ),
        trackeval.metrics.Identity(
            metrics_config
        ),
    ]

    print("=" * 96)
    print("RUNNING TRACKEVAL THROUGH DIRECT PYTHON API")
    print("=" * 96)
    print("TrackEval commit :", commit)
    print("NumPy version    :", np.__version__)
    print("Systems          :", systems)
    print("Sequences        :", SELECTED_SEGMENTS["sequence_name"].tolist())
    print("Output folder    :", TRACKEVAL_OUTPUT_ROOT)
    print("=" * 96)

    start_time = time.perf_counter()

    try:
        TRACK_EVAL_RESULTS = evaluator.evaluate(
            dataset_list,
            metrics_list,
        )

    except Exception:
        error_text = traceback.format_exc()

        (
            TRACKEVAL_DIR
            / "trackeval_direct_api_error.log"
        ).write_text(
            error_text,
            encoding="utf-8",
        )

        raise RuntimeError(
            "TrackEval direct Python API failed. "
            f"See {TRACKEVAL_DIR / 'trackeval_direct_api_error.log'}"
        )

    elapsed_s = (
        time.perf_counter()
        - start_time
    )

    write_json(
        TRACKEVAL_DIR / "trackeval_execution.json",
        {
            "status": "PASSED",
            "execution_mode": "direct_python_api",
            "commit": commit,
            "numpy_version": np.__version__,
            "elapsed_seconds": elapsed_s,
            "systems": systems,
            "sequences": (
                SELECTED_SEGMENTS[
                    "sequence_name"
                ]
                .astype(str)
                .tolist()
            ),
        },
    )

    TRACK_EVAL_EXECUTED = True

    print("=" * 96)
    print("TRACKEVAL: PASSED")
    print(f"Elapsed: {elapsed_s:.2f} s")
    print("=" * 96)

    return True


run_trackeval()

TrackEval NumPy compatibility shim: no missing aliases required.

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 1                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/step5_tools/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
GT_FOLDER            : /content/drive/MyDrive/aerial_human_detection/step5_tracking_final/okutama_l4_final_s42/04_trackeval

True

<Figure size 640x480 with 0 Axes>

## 15. Build the final Step-5 metric matrix

In [40]:
if not globals().get("TRACK_EVAL_EXECUTED", False):
    raise RuntimeError(
        "Final Step-5 metric matrix was not built because TrackEval did not run. "
        "Fix the upstream detector/tracker failure first; see "
        f"{TRACKEVAL_DIR / 'TRACK_EVAL_SKIPPED.json'} if it exists."
    )

def read_trackeval_summary_file(path: Path) -> Dict[str, float]:
    lines = [x.strip() for x in path.read_text(encoding="utf-8", errors="replace").splitlines() if x.strip()]
    if len(lines) < 2: raise RuntimeError(f"Unexpected TrackEval summary: {path}")
    headers, values = lines[0].split(), lines[1].split()
    if len(headers) != len(values): raise RuntimeError(f"Header/value mismatch: {path}")
    out = {}
    for k, v in zip(headers, values):
        try: out[k] = float(v)
        except ValueError: pass
    return out


PERCENT_LIKE_KEYS = {"HOTA","DetA","AssA","LocA","MOTA","MOTP","MODA","CLR_Re","CLR_Pr","MTR","PTR","MLR","sMOTA","IDF1","IDR","IDP"}
def normalize_trackeval_fraction(k, v):
    return v/100.0 if k in PERCENT_LIKE_KEYS and np.isfinite(v) and abs(v) > 1.5 else v


summary_files = sorted(TRACKEVAL_OUTPUT_ROOT.rglob("*_summary.txt"))
if not summary_files:
    summary_files = sorted(TRACKEVAL_TRACKERS_ROOT.rglob("*_summary.txt"))
if not summary_files:
    raise RuntimeError("TrackEval completed but no *_summary.txt files were found")

tracking_rows = []
for path in summary_files:
    system = next((x for x in expected_system_names() if x in str(path)), None)
    if system is None: continue
    metrics = {k: normalize_trackeval_fraction(k, v) for k, v in read_trackeval_summary_file(path).items()}
    tracking_rows.append({"system_name": system, "summary_file": str(path), **metrics})
TRACKING_METRICS = pd.DataFrame(tracking_rows).sort_values("system_name").drop_duplicates("system_name", keep="last").reset_index(drop=True)
if TRACKING_METRICS.empty: raise RuntimeError("Could not map TrackEval summaries to Step-5 systems")

lookup = {safe_slug(f"{m}__{t}"): (m,t) for m in MODELS_TO_RUN for t in TRACKERS_TO_RUN}
TRACKING_METRICS["model"] = TRACKING_METRICS["system_name"].map(lambda x: lookup[x][0])
TRACKING_METRICS["tracker"] = TRACKING_METRICS["system_name"].map(lambda x: lookup[x][1])
if "IDSW" not in TRACKING_METRICS.columns:
    for alt in ["IDs", "IDSw"]:
        if alt in TRACKING_METRICS.columns: TRACKING_METRICS["IDSW"] = TRACKING_METRICS[alt]; break
if "Frag" not in TRACKING_METRICS.columns:
    for alt in ["FRAG", "Fragments"]:
        if alt in TRACKING_METRICS.columns: TRACKING_METRICS["Frag"] = TRACKING_METRICS[alt]; break
if "IDSW" not in TRACKING_METRICS.columns: TRACKING_METRICS["IDSW"] = np.nan
if "Frag" not in TRACKING_METRICS.columns: TRACKING_METRICS["Frag"] = np.nan

RUNTIME_AGG = TRACKER_RUN_SUMMARIES.groupby(
    ["system_name", "model", "tracker"],
    as_index=False,
).agg(
    frames=("frames", "sum"),
    mean_tracker_ms=("mean_tracker_ms", "mean"),
    p50_tracker_ms=("p50_tracker_ms", "mean"),
    p95_tracker_ms=("p95_tracker_ms", "mean"),
    mean_global_identity_ms=("mean_global_identity_ms", "mean"),
    p95_global_identity_ms=("p95_global_identity_ms", "mean"),
    mean_core_pipeline_ms=("mean_core_pipeline_ms", "mean"),
    p50_core_pipeline_ms=("p50_core_pipeline_ms", "mean"),
    p95_core_pipeline_ms=("p95_core_pipeline_ms", "mean"),
    core_pipeline_fps=("core_pipeline_fps", "mean"),
    tracker_overhead_fraction=("tracker_overhead_fraction", "mean"),
    annotated_export_wall_fps=("annotated_export_wall_fps", "mean"),
    detector_peak_cuda_mib=("detector_peak_cuda_mib", "max"),
    recovered_global_id_events=("recovered_global_id_events", "sum"),
    ambiguous_reid_rejections=("ambiguous_reid_rejections", "sum"),
    no_feature_rejections=("no_feature_rejections", "sum"),
    local_to_global_reduction=("local_to_global_reduction", "sum"),
)
FINAL_TRACKING_MATRIX = TRACKING_METRICS.merge(RUNTIME_AGG, on=["system_name","model","tracker"], how="left")
FINAL_TRACKING_MATRIX["idsw_per_100_frames"] = FINAL_TRACKING_MATRIX["IDSW"] / FINAL_TRACKING_MATRIX["frames"].clip(lower=1) * 100
FINAL_TRACKING_MATRIX["fragments_per_100_frames"] = FINAL_TRACKING_MATRIX["Frag"] / FINAL_TRACKING_MATRIX["frames"].clip(lower=1) * 100
FINAL_TRACKING_MATRIX.to_csv(METRICS_DIR/"tracking_metrics_all_systems.csv", index=False)
write_json(METRICS_DIR/"tracking_metrics_all_systems.json", FINAL_TRACKING_MATRIX.to_dict("records"))

GLOBAL_ID_DIAGNOSTICS = RUNTIME_AGG[
    [
        "system_name",
        "model",
        "tracker",
        "recovered_global_id_events",
        "ambiguous_reid_rejections",
        "no_feature_rejections",
        "local_to_global_reduction",
    ]
].copy()

GLOBAL_ID_DIAGNOSTICS.to_csv(
    METRICS_DIR / "global_id_recovery_diagnostics.csv",
    index=False,
)

DETECTOR_AGG = DETECTOR_SUMMARIES.groupby("model", as_index=False).agg(
    frames=("frames","sum"), detector_fps=("detector_fps","mean"), mean_detector_total_ms=("mean_detector_total_ms","mean"),
    p95_detector_total_ms=("p95_detector_total_ms","mean"), peak_cuda_mib=("peak_cuda_mib","max"),
    precision=("precision","mean"), recall=("recall","mean"), f1=("f1","mean"),
    visible_recall=("visible_recall","mean"), occluded_recall=("occluded_recall","mean"),
    h_lt16_recall=("h_lt16_recall","mean"), h_16_31_recall=("h_16_31_recall","mean"),
    h_32_95_recall=("h_32_95_recall","mean"), h_ge96_recall=("h_ge96_recall","mean")
)
DETECTOR_AGG.to_csv(METRICS_DIR/"detector_metrics_aggregate.csv", index=False)

# Raw per-sequence detailed TrackEval tables when available.
detailed = []
for path in TRACKEVAL_OUTPUT_ROOT.rglob("*_detailed.csv"):
    system = next((x for x in expected_system_names() if x in str(path)), None)
    if system is None: continue
    try: df = pd.read_csv(path)
    except Exception: continue
    df["system_name"] = system; df["source_file"] = str(path); detailed.append(df)
if detailed:
    pd.concat(detailed, ignore_index=True, sort=False).to_csv(METRICS_DIR/"tracking_metrics_per_sequence_raw.csv", index=False)

# BoT-SORT minus ByteTrack delta per detector.
delta_rows = []
for model_name in MODELS_TO_RUN:
    d = FINAL_TRACKING_MATRIX[FINAL_TRACKING_MATRIX["model"] == model_name].set_index("tracker")
    if not {"ByteTrack","BoT-SORT"}.issubset(d.index): continue
    bt, bs = d.loc["ByteTrack"], d.loc["BoT-SORT"]; rec = {"model": model_name}
    for metric in ["HOTA","DetA","AssA","MOTA","MOTP","IDF1","IDSW","Frag","idsw_per_100_frames"]:
        if metric in d.columns: rec[f"delta_botsort_minus_bytetrack__{metric}"] = float(bs[metric])-float(bt[metric])
    delta_rows.append(rec)
TRACKER_DELTA = pd.DataFrame(delta_rows); TRACKER_DELTA.to_csv(METRICS_DIR/"tracker_delta_botsort_minus_bytetrack.csv", index=False)

show_cols = [
    c
    for c in [
        "model",
        "tracker",
        "HOTA",
        "DetA",
        "AssA",
        "MOTA",
        "MOTP",
        "IDF1",
        "IDSW",
        "Frag",
        "idsw_per_100_frames",
    ]
    if c in FINAL_TRACKING_MATRIX.columns
]
display(FINAL_TRACKING_MATRIX[show_cols].sort_values(["model","tracker"]).reset_index(drop=True))


,model,tracker,HOTA,DetA,AssA,MOTA,MOTP,IDF1,IDSW,Frag,idsw_per_100_frames
0,BPD-YOLOn/L-FPN,BoT-SORT,0.19586,0.14613,0.27110,-0.21304,0.63252,0.21538,39.0,232.0,2.600000
1,BPD-YOLOn/L-FPN,ByteTrack,0.19782,0.14552,0.27472,-0.18980,0.63526,0.21954,29.0,213.0,1.933333
2,RT-DETR-R18,BoT-SORT,0.26585,0.19272,0.37111,-0.34512,0.62257,0.29064,19.0,207.0,1.266667
3,RT-DETR-R18,ByteTrack,0.26222,0.19463,0.35771,-0.33201,0.62625,0.28489,21.0,204.0,1.400000
4,YOLO26s,BoT-SORT,0.24502,0.17130,0.35773,-0.24367,0.63678,0.28123,10.0,312.0,0.666667
5,YOLO26s,ByteTrack,0.22426,0.17203,0.29934,-0.23232,0.64013,0.25071,22.0,309.0,1.466667


## 16. Proposal-reference acceptance checks and report-ready plots
These checks report the project gates; they are not used to tune the Okutama test clips.

In [41]:
acceptance_rows = []

for _, row in FINAL_TRACKING_MATRIX.iterrows():
    mota = float(
        row.get(
            "MOTA",
            np.nan,
        )
    )

    idf1 = float(
        row.get(
            "IDF1",
            np.nan,
        )
    )

    idsw100 = float(
        row.get(
            "idsw_per_100_frames",
            np.nan,
        )
    )

    rec = {
        "system_name": row["system_name"],
        "model": row["model"],
        "tracker": row["tracker"],

        "MOTA": mota,
        "target_MOTA": TARGET_MOTA,
        "pass_MOTA": bool(
            np.isfinite(mota)
            and mota >= TARGET_MOTA
        ),

        "IDF1": idf1,
        "target_IDF1": TARGET_IDF1,
        "pass_IDF1": bool(
            np.isfinite(idf1)
            and idf1 >= TARGET_IDF1
        ),

        "IDSW_per_100_frames": idsw100,
        "target_max_IDSW_per_100_frames": (
            TARGET_IDSW_PER_100_FRAMES
        ),
        "pass_IDSW": bool(
            np.isfinite(idsw100)
            and idsw100
            <= TARGET_IDSW_PER_100_FRAMES
        ),
    }

    rec[
        "pass_all_reference_checks"
    ] = all(
        [
            rec["pass_MOTA"],
            rec["pass_IDF1"],
            rec["pass_IDSW"],
        ]
    )

    acceptance_rows.append(
        rec
    )


ACCEPTANCE = pd.DataFrame(
    acceptance_rows
)

ACCEPTANCE.to_csv(
    METRICS_DIR
    / "proposal_reference_acceptance_checks.csv",
    index=False,
)

def system_labels(df): return [f"{m}\n{t}" for m,t in zip(df["model"], df["tracker"])]
def save_bar(metric, ylabel, filename):
    if metric not in FINAL_TRACKING_MATRIX.columns: return
    df = FINAL_TRACKING_MATRIX.sort_values(["model","tracker"]).reset_index(drop=True)
    values = pd.to_numeric(df[metric], errors="coerce").to_numpy()
    fig, ax = plt.subplots(figsize=(12,6)); ax.bar(system_labels(df), values); ax.set_ylabel(ylabel)
    ax.set_title(f"Step-5 Tracking Comparison — {metric}"); ax.tick_params(axis="x", rotation=25); ax.grid(axis="y", alpha=.25)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/filename, dpi=180); plt.close(fig)


save_bar("HOTA","HOTA","01_hota_comparison.png"); save_bar("IDF1","IDF1","02_idf1_comparison.png")
save_bar("MOTA","MOTA","03_mota_comparison.png"); save_bar("idsw_per_100_frames","ID switches / 100 frames","04_idsw_per_100_frames.png")




if not DETECTOR_AGG.empty:
    plot_df = DETECTOR_AGG.set_index("model")[["recall","visible_recall","occluded_recall"]]
    fig, ax = plt.subplots(figsize=(10,6)); plot_df.plot(kind="bar", ax=ax); ax.set_ylabel("Recall at IoU operating point")
    ax.set_title("Detector Recall on Selected Okutama Clips"); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.25)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/"09_detector_visible_occluded_recall.png", dpi=180); plt.close(fig)
    size_df = DETECTOR_AGG.set_index("model")[["h_lt16_recall","h_16_31_recall","h_32_95_recall","h_ge96_recall"]]
    fig, ax = plt.subplots(figsize=(11,6)); size_df.plot(kind="bar", ax=ax); ax.set_ylabel("Recall at IoU operating point")
    ax.set_title("Detector Recall by GT Person Height"); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.25)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/"10_detector_size_aware_recall.png", dpi=180); plt.close(fig)

display(ACCEPTANCE)


if (
    "recovered_global_id_events"
    in FINAL_TRACKING_MATRIX.columns
):
    save_bar(
        "recovered_global_id_events",
        "Recovered Global IDs",
        "05_global_id_recoveries.png",
    )


,system_name,model,tracker,MOTA,target_MOTA,pass_MOTA,IDF1,target_IDF1,pass_IDF1,IDSW_per_100_frames,target_max_IDSW_per_100_frames,pass_IDSW,pass_all_reference_checks
0,BPD-YOLOn_L-FPN_BoT-SORT,BPD-YOLOn/L-FPN,BoT-SORT,-0.21304,0.5,False,0.21538,0.6,False,2.600000,5.0,True,False
1,BPD-YOLOn_L-FPN_ByteTrack,BPD-YOLOn/L-FPN,ByteTrack,-0.18980,0.5,False,0.21954,0.6,False,1.933333,5.0,True,False
2,RT-DETR-R18_BoT-SORT,RT-DETR-R18,BoT-SORT,-0.34512,0.5,False,0.29064,0.6,False,1.266667,5.0,True,False
3,RT-DETR-R18_ByteTrack,RT-DETR-R18,ByteTrack,-0.33201,0.5,False,0.28489,0.6,False,1.400000,5.0,True,False
4,YOLO26s_BoT-SORT,YOLO26s,BoT-SORT,-0.24367,0.5,False,0.28123,0.6,False,0.666667,5.0,True,False
5,YOLO26s_ByteTrack,YOLO26s,ByteTrack,-0.23232,0.5,False,0.25071,0.6,False,1.466667,5.0,True,False


## 17. Automatic Step-5 report package and SHA-256 manifest

In [42]:
def best_row(metric, maximize=True):
    if metric not in FINAL_TRACKING_MATRIX.columns: return None
    vals = pd.to_numeric(FINAL_TRACKING_MATRIX[metric], errors="coerce")
    if not vals.notna().any(): return None
    return FINAL_TRACKING_MATRIX.loc[vals.idxmax() if maximize else vals.idxmin()]


leaders = {}
for metric, maximize in [("HOTA",True),("IDF1",True),("MOTA",True),("IDSW",False)]:
    r = best_row(metric, maximize)
    if r is not None: leaders[metric] = {"system_name": r["system_name"], "model": r["model"], "tracker": r["tracker"], "value": float(r[metric])}

expected_system_count = len(MODELS_TO_RUN)*len(TRACKERS_TO_RUN); actual_system_count = FINAL_TRACKING_MATRIX["system_name"].nunique()
expected_sequence_count = len(SELECTED_SEGMENTS); expected_run_count = expected_system_count*expected_sequence_count; actual_run_count = len(TRACKER_RUN_SUMMARIES)
completeness = {"expected_systems": expected_system_count, "evaluated_systems": int(actual_system_count),
                "expected_sequences": expected_sequence_count, "expected_tracker_segment_runs": expected_run_count,
                "completed_tracker_segment_runs": int(actual_run_count), "system_errors": len(SYSTEM_ERRORS),
                "complete": bool(actual_system_count==expected_system_count and actual_run_count==expected_run_count and not SYSTEM_ERRORS)}
write_json(AUDIT_DIR/"completion_audit.json", completeness)

FINAL_TRACKING_MATRIX.to_csv(REPORT_ASSETS_DIR/"table_tracking_main.csv", index=False)
DETECTOR_AGG.to_csv(REPORT_ASSETS_DIR/"table_detector_component.csv", index=False)
TRACKER_DELTA.to_csv(REPORT_ASSETS_DIR/"table_tracker_delta.csv", index=False)
GLOBAL_ID_DIAGNOSTICS.to_csv(
    REPORT_ASSETS_DIR / "table_global_id_diagnostics.csv",
    index=False,
)
ACCEPTANCE.to_csv(REPORT_ASSETS_DIR/"table_acceptance_checks.csv", index=False)
SELECTED_SEGMENTS.to_csv(REPORT_ASSETS_DIR/"table_selected_okutama_clips.csv", index=False)

report_cols = [
    c
    for c in [
        "model",
        "tracker",
        "HOTA",
        "DetA",
        "AssA",
        "MOTA",
        "MOTP",
        "IDF1",
        "IDSW",
        "Frag",
        "idsw_per_100_frames",
    ]
    if c in FINAL_TRACKING_MATRIX.columns
]
report_table = FINAL_TRACKING_MATRIX[report_cols].sort_values(["model","tracker"]).reset_index(drop=True)
lines = [
    "# STEP 5 — Multi-Object Tracking Integration Report", "", f"Generated: {utc_now_iso()}", "",
    "## Scope", "", "- Three fixed final aerial-person detectors.", "- ByteTrack and BoT-SORT with the same external Long-Term Global-ID ReID layer.",
    "- Identical cached detector outputs are replayed into both trackers.", "- Short gaps use a 3-second tracker buffer; longer FOV exits use a 60-second Global-ID appearance gallery.",
    "- Okutama-Action continuous aerial clips with reference person IDs.",
    "- HOTA/CLEAR/Identity metrics are computed with TrackEval.", "- FPS/latency are retained only as internal diagnostics; Step 4 is the deployment-performance benchmark.", "- No private project Final Test is used.",
    "- Tracker/ReID settings were frozen after the earlier Okutama pilot; final unbiased claims require fresh evaluation clips.", "", "## Environment", "",
    f"- GPU: {GPU_NAME}", f"- PyTorch: {torch.__version__}", f"- CUDA: {torch.version.cuda}",
    f"- Ultralytics: {ultralytics.__version__}", "", "## Okutama protocol", "",
    f"- Dataset profile: {DATASET_PROFILE}", f"- Selected videos: {SELECTED_SEGMENTS['video_key'].nunique()}",
    f"- Selected clips: {len(SELECTED_SEGMENTS)}", f"- Label mode(s): {', '.join(sorted(SELECTED_SEGMENTS['label_mode'].unique()))}",
    f"- Consistent tracking IDs available: {bool(SELECTED_SEGMENTS['consistent_ids'].all())}", "",
    "SingleActionTrackingLabels are preferred. If fallback labels are used, windows are capped at 180 frames to avoid the official ID-reset boundary.",
    "", "## Main tracking matrix", "", report_table.to_markdown(index=False), "", "## Metric leaders (descriptive only)", ""
]
for metric, info in leaders.items(): lines.append(f"- {metric}: {info['model']} + {info['tracker']} = {info['value']:.4f}")
lines += [
    "",
    "These leaders are not a final product winner; Step 6 performs the final multi-criteria comparison.",
    "",
    "## Global-ID recovery diagnostics",
    "",
    GLOBAL_ID_DIAGNOSTICS.to_markdown(index=False),
    "",
          "## Completion audit", "", "```json", json.dumps(completeness, indent=2), "```", "",
          "## Important interpretation", "", "Step 5 measures detector-to-tracker integration and identity/association behavior on continuous aerial video. It does not replace the later independent final detection benchmark or the final RTX 3070 deployment benchmark.", ""]
(OUTPUT_ROOT/"STEP5_RUN_REPORT.md").write_text("\n".join(lines), encoding="utf-8")
write_json(OUTPUT_ROOT/"STEP5_RUN_REPORT.json", {
    "generated_utc": utc_now_iso(), "output_root": str(OUTPUT_ROOT), "environment": environment,
    "execution_config": execution_config, "dataset_summary": dataset_summary, "leaders": leaders,
    "completeness": completeness, "main_metrics": FINAL_TRACKING_MATRIX.to_dict("records"),
    "detector_metrics": DETECTOR_AGG.to_dict("records"), "tracker_delta": TRACKER_DELTA.to_dict("records"),
    "acceptance_checks": ACCEPTANCE.to_dict("records"),
})

readme = f'''# Step-5 Delivery Package

Persistent output root:
`{OUTPUT_ROOT}`

Most important report files:
1. STEP5_RUN_REPORT.md
2. 05_metrics/tracking_metrics_all_systems.csv
3. 05_metrics/detector_metrics_aggregate.csv
4. 05_metrics/tracker_delta_botsort_minus_bytetrack.csv
5. 05_metrics/proposal_reference_acceptance_checks.csv\n6. 05_metrics/global_id_recovery_diagnostics.csv
6. 02_dataset/selected_segments.csv
7. 06_plots/*.png
8. 03_runs/*/*/annotated_tracking.mp4
9. 04_trackeval/trackeval_console.log
10. 00_audit/environment.json
11. 01_models/model_source_manifest.csv
12. MANIFEST_SHA256.csv

Do not report a final project winner from Step 5 alone.
'''
(OUTPUT_ROOT/"README_OUTPUTS.md").write_text(readme, encoding="utf-8")

manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if not path.is_file() or path.name == "MANIFEST_SHA256.csv": continue
    try: digest = sha256_file(path)
    except Exception as exc: digest = f"ERROR:{exc}"
    manifest_rows.append({"relative_path": str(path.relative_to(OUTPUT_ROOT)), "size_bytes": path.stat().st_size, "sha256": digest})
MANIFEST = pd.DataFrame(manifest_rows); MANIFEST.to_csv(OUTPUT_ROOT/"MANIFEST_SHA256.csv", index=False)

print("="*100); print("STEP 5 FINAL PACKAGE CREATED"); print("="*100)
print("Output root:", OUTPUT_ROOT); print("Systems:", actual_system_count, "/", expected_system_count)
print("Run matrix:", actual_run_count, "/", expected_run_count); print("Errors:", len(SYSTEM_ERRORS)); print("Complete:", completeness["complete"])
print("="*100)
if FAIL_IF_FINAL_MATRIX_INCOMPLETE and not completeness["complete"]:
    raise RuntimeError("Outputs were saved, but the Step-5 matrix is incomplete. Inspect 00_audit/system_errors.json and rerun.")


STEP 5 FINAL PACKAGE CREATED
Output root: /content/drive/MyDrive/aerial_human_detection/step5_tracking_final/okutama_l4_final_s42
Systems: 6 / 6
Run matrix: 30 / 30
Errors: 0
Complete: True


## Final configuration summary

- Detector confidence: YOLO26s `0.20`, RT-DETR-R18 `0.45`, BPD-YOLOn/L-FPN `0.20`
- Short-term lost-track retention: **3 seconds**
- Long-term Global-ID memory: **60 seconds**
- Long-term ReID encoder: `yolo26n-reid.onnx`
- Conservative global recovery gates: cosine `>= 0.88`, combined score `>= 0.83`,
  best-candidate margin `>= 0.06`
- BoT-SORT internal ReID: enabled
- BoT-SORT GMC: `sparseOptFlow`
- Formal TrackEval input: **Global IDs**
- Raw Local tracker IDs: saved separately for audit
- Step-5 formal acceptance: MOTA / IDF1 / IDSW only
